# TabTune · Home Credit · Loan Amount Recommendation
### Tabular foundation models, end to end: data preparation → AutoML → testing → risk → production

**How much credit should this applicant be offered?** A retail lender recommends a loan amount from the applicant's profile, bureau history and previous applications, to support limit setting and affordability checks.

| | |
|---|---|
| **Task** | Regression: credit amount (`AMT_CREDIT`) |
| **Data** | Home Credit applications, about 270 features (application, bureau, previous loans, engineered ratios) |
| **Decision supported** | Recommended amount with a safe range |
| **Engine** | TabTune open-source library, running on this notebook's GPU |
| **Production route** | The Lexsi platform (reference cells show the equivalent calls) |

## Notebook map

| # | Section | What you get |
|---|---|---|
| 1 | Onboarding | Install, model options in TabTune and on the Lexsi platform |
| 2 | Product options | Problem statement, training and test data, development vs production choices |
| 3 | Data preparation | Data profile dashboards, a **data preparation agent** with approval gates, feature engineering, conclusions |
| 4 | Model training | AutoML race (foundation models vs tuned classic models), hyperparameter search, selection rule, global explainability |
| 5 | Model testing | Evaluation dashboards on held-out data, interactive threshold, per-decision explanations |
| 6 | Batch inference | Scoring a file end to end, throughput, saved model artefact |
| 7 | Model risk | Uncertainty with guaranteed coverage, fairness, regulatory evidence map |
| 8 | Artefacts before production | Model card written through the **Lexsi LLM gateway** from a documented prompt, with automatic hallucination checks and a revision pass; production API contract |
| 9 | Model in production | Five monitors (data drift, prediction drift, performance, calibration, data quality), alert rules, a **webhook** that receives alerts and hands them to the **retraining agent**, regenerated model card |
| 10 | Conclusions | Plain-English summary and the audit trail |

**How to read it:** the short visible cells are the calls that matter (`TabularPipeline`, `fit`, `predict`, `evaluate`, `save`, conformal `calibrate`, the agents). Collapsed cells (▸ *Show code*) hold plotting and helper code; their output still shows.

**Run:** *Runtime → Change runtime type → A100 (or H100) GPU, High-RAM*, then *Run all*. Set `AUTO_APPROVE = False` in the imports cell to be asked at each approval gate. For the model card, add a `LEXSI_API_TOKEN` Colab secret and set the provider and model in section 8.1 (the card is written through the Lexsi LLM gateway).

# 1 · Onboarding

In [ ]:
# @title Install TabTune and dependencies { display-mode: "form" }
# ⚙️ Install TabTune (open source) — about 3 minutes on Colab
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError

def _v(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

def pip_install(*packages):
    # pip install that prints pip's own error message instead of a bare CalledProcessError
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-4000:])
        raise RuntimeError("pip install failed; pip's message is printed above")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    WATCH = {"numpy": "numpy", "pandas": "pandas", "scikit-learn": "sklearn", "transformers": "transformers", "huggingface-hub": "huggingface_hub"}
    before = {p: _v(p) for p in WATCH}
    # One pip call, so pip resolves everything together:
    #  * no [colab] extra: it pins pandas==2.2.2, which has no wheel for Python 3.13
    #  * TabTune needs transformers 4.x, and transformers 4.x needs huggingface-hub < 1.0 (Colab ships 2.x)
    #  * the Lexsi SDK is not installed: platform calls are shown as reference text only, and its
    #    OpenTelemetry dependencies change Colab's protobuf version
    pip_install("tabtune @ git+https://github.com/Lexsi-Labs/TabTune.git", "huggingface_hub>=0.34,<1.0",
                "plotly", "ipywidgets", "accelerate", "kernels>=0.9,<0.11", "httpx[http2]")
    # Changing a package that this Python session has already imported needs a restart
    stale = [p for p, mod in WATCH.items() if before[p] != _v(p) and mod in sys.modules]
    if stale:
        raise SystemExit(f"⚠️ {', '.join(stale)} changed after being imported. Runtime → Restart session, then Run all (the reinstall is instant).")
    # Check the stack imports cleanly in a fresh process before going on
    chk = subprocess.run([sys.executable, "-c", "from tabtune import TabularPipeline; import transformers, huggingface_hub; "
                          "print(transformers.__version__, huggingface_hub.__version__)"], capture_output=True, text=True)
    if chk.returncode != 0:
        print(chk.stderr[-3000:])
        raise RuntimeError("TabTune does not import cleanly; the error is printed above")

print(f"python       {sys.version.split()[0]}")
for pkg in ("tabtune", "torch", "transformers", "huggingface-hub", "numpy", "pandas", "kernels", "httpx", "h2"):
    print(f"{pkg:12} {_v(pkg) or 'MISSING'}")

In [ ]:
# @title Imports, theme, fact sheet, audit trail and approval gates { display-mode: "form" }
# 🔧 Imports, theme and the run-wide fact sheet
import json, logging, re, time, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML
import ipywidgets as W

warnings.filterwarnings("ignore")
from tabtune.logger import setup_logger
setup_logger(level=logging.WARNING)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 60)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none"
except ImportError:
    DEVICE, GPU = "cpu", "none"

FAST = os.environ.get("TABTUNE_DEMO_FAST") == "1"      # tiny settings for a smoke test without a GPU
AUTO_APPROVE = True                                    # False = the agents stop and ask you to type yes / no
BLUE, ORANGE, TEAL, GREY, DARK, RED, GOLD = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D", "#E8C547"
PALETTE = [BLUE, ORANGE, TEAL, RED, GOLD, GREY, DARK, "#7A5CFA"]
FACTS = {}                                             # every number that reaches the model card is recorded here
AUDIT = []                                             # every agent step and human decision


def record(section, **values):
    """Store results for the model card and the conclusions."""
    FACTS.setdefault(section, {}).update({k: (round(float(v), 4) if isinstance(v, (int, float, np.floating, np.integer)) and not isinstance(v, bool) else v)
                                          for k, v in values.items()})


def audit(actor, step, finding, decision=""):
    AUDIT.append({"at": pd.Timestamp.now().strftime("%H:%M:%S"), "actor": actor, "step": step, "finding": finding, "decision": decision})
    print(f"[{actor} · {step}] {finding}" + (f"  →  {decision}" if decision else ""))


def human_gate(question, recommended=True):
    """Human-in-the-loop checkpoint. With AUTO_APPROVE the recommendation is taken and logged; otherwise you are asked."""
    if AUTO_APPROVE:
        answer = recommended
        who = "auto-approved (AUTO_APPROVE=True)"
    else:
        reply = input(f"🧑 {question}  [recommended: {'yes' if recommended else 'no'}]  yes/no: ").strip().lower()
        answer = reply.startswith("y") if reply else recommended
        who = "human reviewer"
    audit("human", "approval gate", question, f"{'APPROVED' if answer else 'REJECTED'} · {who}")
    return answer


print(f"device: {DEVICE} ({GPU}) · fast mode: {FAST} · auto-approve: {AUTO_APPROVE}")

In [ ]:
# @title Model options in TabTune today (read live from the library's registry) { display-mode: "form" }
from tabtune import TabularPipeline
from tabtune.registry import list_models
import logging
for _n in list(logging.root.manager.loggerDict):          # keep the library's progress log out of the demo output
    if "tabtune" in _n.lower():
        logging.getLogger(_n).setLevel(logging.WARNING)

TASK = "regression"
def _lic(spec):
    lic = getattr(spec, "license", None)
    return getattr(lic, "name", "?"), {True: "yes", False: "no", None: "unverified"}[getattr(lic, "commercial_use_ok", None)]

TABTUNE_MODELS = {}
for spec in list_models():
    env = getattr(spec, "envelope", None)
    TABTUNE_MODELS[spec.name] = {
        "family": getattr(spec, "family", ""),
        "classification": bool(spec.classification_strategies), "regression": bool(spec.regression_strategies),
        "strategies": ", ".join(sorted(getattr(spec, f"{TASK}_strategies"))),
        "max_rows": getattr(env, "max_rows", None), "max_features": getattr(env, "max_features", None),
        "max_classes": getattr(env, "max_classes", None),
        "licence": _lic(spec)[0], "commercial_use": _lic(spec)[1], "summary": (spec.summary or "")[:70]}
catalogue = pd.DataFrame(TABTUNE_MODELS).T.rename_axis("model")
usable = catalogue[catalogue[TASK]]
print(f"{len(catalogue)} tabular foundation models registered · {len(usable)} support {TASK} · "
      f"{int((usable.commercial_use == 'yes').sum())} of those are cleared for commercial use")
display(usable.drop(columns=["classification", "regression"]).style.map(
    lambda v: f"color:{TEAL};font-weight:600" if v == "yes" else (f"color:{RED}" if v == "no" else ""), subset=["commercial_use"]))

### The same options on the Lexsi platform
Development happens here with the open-source library. In production the same models run on the Lexsi platform, which adds projects, managed compute, a model registry, case-level audit and policies.

In [ ]:
# @title The same workflow on the Lexsi platform (reference: nothing in this cell calls the platform) { display-mode: "form" }
LEXSI_OPTIONS = {
    "structure": "Organization → Workspace → Project (one project per use case; project type is classification or regression)",
    "data": "project.upload_data(file, tag): tags such as Training / Test / Batch1 select data for training, testing and inference",
    "training_strategies": ["simple", "ensemble", "distillation", "leaderboard"],
    "tuning_strategies": ["inference (zero-shot)", "finetune", "peft (LoRA)"],
    "compute": {"CPU": ["xsmall", "small", "medium", "large", "xlarge", "2xlarge", "3xlarge"],
                "GPU": ["T4.xsmall", "T4.small", "A10G.xmedium"]},
    "foundation_models": ["TabPFN", "TabPFNv26", "TabPFNv3", "TabICL", "TabICLv2", "OrionMSP", "OrionMSPv1.5", "OrionBix", "OrionMoE",
                          "Mitra", "ContextTab", "TabDPT", "Limix", "TabFM", "EXAONETabular", "ILTM", "XRFM"],
    "classic_models": ["XGBoost", "LGBoost", "CatBoost", "RandomForest", "SGD", "LogisticRegression", "LinearRegression", "GaussianNaiveBayes"],
    "explainability": ["shap", "lime", "support_set (training-row attribution, in-context models)"],
    "inference": "project.model_inference(tag=..., model_name=..., compute_type=...) for batches; project.case_predict(...) for one case",
    "governance": "model registry per project, activate_model for promotion, case audit trail, policies and observations",
}
display(pd.DataFrame({"On the Lexsi platform": {k: (", ".join(v) if isinstance(v, list) else json.dumps(v) if isinstance(v, dict) else v)
                                               for k, v in LEXSI_OPTIONS.items()}}).style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
display(Markdown("""
**Platform equivalent of this notebook's training step** (reference only; run it in a separate environment after `pip install lexsi-sdk`):
```python
from lexsi_sdk import lexsi
lexsi.login()
project = lexsi.organization("…").workspace("…").project("…")
project.upload_data("train.csv", "Training", config={"unique_identifier": "id", "true_label": "TARGET"}, compute_type="small")
project.train_model(model="OrionMSP", training_strategy="simple", tunning_strategy="inference", compute_type="T4.small",
                    data_config={"tags": ["Training"], "test_tags": ["Test"], "xai_method": ["shap", "lime"]})
```"""))

### LLM for the agents and the model card
The agents in this notebook are **LLM agents**. Each one gets a goal, a set of **tools** (Python functions that inspect the data or act on it), and the evidence those tools return. The LLM decides which tool to call next and why; the tool runs; the result goes back to the LLM; and so on until it calls `finish`.

- **The LLM** is the model set below (`LEXSI_PROVIDER`, `LEXSI_MODEL`), called through the **Lexsi LLM gateway** (`/gateway/v1/chat/completions`), so every call is traced in the Lexsi project. The same model writes the model card in section 8.
- **A human approves** every action that changes data or models. The approval prompt shows the agent's own reasoning.
- **Rules guard the result.** After an agent finishes, hard rules check what must hold (for example: known leaks removed, no series too short to model). If the agent left something undone, the rule applies it and says so.
- **Without the gateway** (no `LEXSI_API_TOKEN` secret), or with `AGENT_MODE = "rules"`, the original rule-based agents run instead.

Only aggregate evidence reaches the LLM: feature names, counts, scores. No customer rows are sent.

In [ ]:
# @title LLM connection (Lexsi gateway) and writer options { display-mode: "form" }
import html as _html, textwrap

GATEWAY_PROVIDERS = ["OpenAI", "Anthropic", "Gemini", "Groq", "Grok", "Mistral", "Hugging Face", "AWS Bedrock", "Replicate",
                     "Open Router", "Together", "Lexsi", "Self Hosted"]          # spelled exactly as the gateway expects
LOCAL_WRITERS = ["openai/gpt-oss-120b", "Qwen/Qwen2.5-32B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]   # fallback: open weights on this GPU
REVISION_PASS = True            # if the automatic checks find problems, send them back to the writer once
try:
    import h2                   # HTTP/2 support for httpx
    HTTP2 = True
except ImportError:
    HTTP2 = False


def _secret(name):
    # Colab secret (key icon in the left bar) or environment variable
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name)


def resolve_writer(choice):
    token = _secret("LEXSI_API_TOKEN")
    if choice == "template":
        return "template"
    if choice in ("auto", "lexsi"):
        missing = [n for n, v in (("LEXSI_API_TOKEN secret", token), ("LEXSI_PROVIDER", LEXSI_PROVIDER), ("LEXSI_MODEL", LEXSI_MODEL)) if not v]
        if not missing and LEXSI_PROVIDER not in GATEWAY_PROVIDERS:
            missing = [f"a valid LEXSI_PROVIDER (got '{LEXSI_PROVIDER}'; use one of: {', '.join(GATEWAY_PROVIDERS)})"]
        if not missing:
            return "lexsi"
        if choice == "lexsi" or token:
            print("Lexsi gateway not used; still needed:", "; ".join(missing))
    if choice in ("auto", "lexsi", "local"):
        return "local" if (DEVICE == "cuda" and not FAST) else "template"
    return choice                                   # an explicit local model name


def writer_options(selected):
    tab = pd.DataFrame([
        ("lexsi", "Lexsi LLM gateway", f"{LEXSI_PROVIDER or '<provider>'} · {LEXSI_MODEL or '<model>'} · project {LEXSI_PROJECT}; every call is traced in the project; "
                  "only the aggregate fact sheet is sent, never customer rows"),
        ("local", "Open weights on this GPU", "gpt-oss-120b (about 63 GB, needs an 80 GB GPU), then Qwen2.5-32B, then Qwen2.5-7B"),
        ("template", "Deterministic", "no LLM; fills the same template from the facts"),
    ], columns=["writer", "kind", "notes"])
    row = selected if selected in ("lexsi", "template") else "local"
    tab[""] = np.where(tab.writer.eq(row), "◀ this run", "")
    return tab.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

In [ ]:
CARD_WRITER = "auto"                           # "auto" = Lexsi gateway if the token secret is set, else local GPU, else template
LEXSI_PROVIDER = "Open Router"                 # exactly as registered in the Lexsi project
LEXSI_MODEL = "anthropic-claude-sonnet-5.5_v1"
LEXSI_PROJECT = "demo_Z1P8OHR7RU"
LEXSI_CLIENT_ID = "adityatanna"
LEXSI_ENDPOINT = "chat"                        # "chat" = /gateway/v1/chat/completions · "text" = /gateway/v1/text/generation (Lexsi-hosted models only)
LEXSI_MAX_TOKENS = 32000                       # thinking tokens count against this limit
LEXSI_REASONING = {"effort": "low"}            # forwarded to OpenRouter: short thinking, so the budget goes to the card · None = model default
LEXSI_TEMPERATURE, LEXSI_TOP_P, LEXSI_TOP_K = 0.2, 0.9, 50     # used by the text endpoint only
LEXSI_MIN_TOKENS, LEXSI_EXPLAIN_TOKENS = None, 1               # used by the text endpoint only
# The API token is read from the Colab secret LEXSI_API_TOKEN (key icon in the left bar); never paste it here.

WRITER = resolve_writer(CARD_WRITER)
writer_options(WRITER)

In [ ]:
import httpx

LEXSI_BASE = os.environ.get("LEXSI_GATEWAY_BASE", "https://api.lexsi.ai/gateway/v1")


def _lexsi_post(path, payload):
    headers = {"x-api-token": _secret("LEXSI_API_TOKEN"), "Content-Type": "application/json"}
    with httpx.Client(http2=HTTP2, timeout=httpx.Timeout(900, connect=30)) as client:
        return client.post(f"{LEXSI_BASE}/{path}", headers=headers, json=payload)


def lexsi_chat(messages, max_tokens, reasoning=None):              # POST /gateway/v1/chat/completions
    payload = {"provider": LEXSI_PROVIDER, "client_id": LEXSI_CLIENT_ID, "project_name": LEXSI_PROJECT, "model": LEXSI_MODEL,
               "max_tokens": max_tokens, "messages": messages, "stream": False}
    if reasoning:
        payload["reasoning"] = reasoning
    return _lexsi_post("chat/completions", payload)


def lexsi_generate(prompt, max_tokens):                            # POST /gateway/v1/text/generation
    payload = {"provider": LEXSI_PROVIDER, "client_id": LEXSI_CLIENT_ID, "project_name": LEXSI_PROJECT, "model": LEXSI_MODEL,
               "prompt": prompt, "explain_tokens": LEXSI_EXPLAIN_TOKENS, "temperature": LEXSI_TEMPERATURE,
               "top_p": LEXSI_TOP_P, "top_k": LEXSI_TOP_K, "max_tokens": max_tokens}
    if LEXSI_MIN_TOKENS:
        payload["min_tokens"] = LEXSI_MIN_TOKENS
    return _lexsi_post("text/generation", payload)

In [ ]:
# @title Agent framework: the LLM decides, tools act, rules guard, a human approves { display-mode: "form" }
AGENT_STEPS, AGENT_CALLS = [], []          # every step every agent took · every LLM call
AGENT_REASONING = {"effort": "low"}        # short thinking per step; set None for the model's default


class Tool:
    def __init__(self, name, fn, description, args=None, approval=False):
        self.name, self.fn, self.description, self.args, self.approval = name, fn, description, args or {}, approval

    def signature(self):
        a = ", ".join(f"{k}: {v}" for k, v in self.args.items())
        return f"- {self.name}({a}) — {self.description}" + (" [needs human approval]" if self.approval else "")


PROTOCOL = """AGENT PROTOCOL
Reply with ONE JSON object and nothing else:
{"thought": "<one or two sentences for a human reviewer: what the evidence shows and why this is the next action>", "tool": "<tool name>", "args": {<arguments>}}
After each action you receive an OBSERVATION with the tool's result. Call the tool "finish" with {"summary": "..."} when the goal is met."""


def _compact(obj, limit=3500):
    """JSON for an observation, shortened by trimming long lists rather than cutting the text."""
    text = json.dumps(obj, default=str)
    keep = 25
    while len(text) > limit and keep >= 3:
        def trim(o):
            if isinstance(o, list) and len(o) > keep:
                return [trim(x) for x in o[:keep]] + [f"... {len(o) - keep} more"]
            if isinstance(o, list):
                return [trim(x) for x in o]
            if isinstance(o, dict):
                return {k: trim(v) for k, v in o.items()}
            return o
        text = json.dumps(trim(obj), default=str)
        keep //= 2
    return text[:limit]


def parse_action(text):
    """First JSON object in the reply that names a tool (tolerates code fences and surrounding prose)."""
    dec = json.JSONDecoder()
    for i, ch in enumerate(text or ""):
        if ch == "{":
            try:
                obj, _ = dec.raw_decode(text[i:])
            except ValueError:
                continue
            if isinstance(obj, dict) and ("tool" in obj or "action" in obj):
                obj.setdefault("tool", obj.pop("action", None))
                obj.setdefault("args", {})
                obj.setdefault("thought", "")
                return obj
    return None


class LLMAgent:
    def __init__(self, name, role, goal, rules, tools, max_steps=16):
        self.name, self.role, self.goal, self.rules, self.max_steps = name, role, goal, rules, max_steps
        self.tools = {t.name: t for t in tools}
        self.summary = None

    def system_prompt(self):
        tools = "\n".join(t.signature() for t in self.tools.values()) + "\n- finish(summary: what you did and why, in plain English) — end the task"
        rules = "\n".join(f"- {r}" for r in ["Decide only from evidence returned by the tools. Never invent numbers, features or series names.",
                                              "Actions marked [needs human approval] are shown to a reviewer together with your thought.",
                                              *self.rules, f"You have at most {self.max_steps} replies."])
        return f"You are the {self.name}. {self.role}\n\nGOAL\n{self.goal}\n\nTOOLS (call exactly one per reply)\n{tools}\n\nRULES\n{rules}\n\n{PROTOCOL}"

    def _llm(self, messages):
        global AGENT_REASONING
        t0 = time.time()
        r = lexsi_chat(messages, AGENT_MAX_TOKENS, AGENT_REASONING)
        if r.status_code in (400, 422) and AGENT_REASONING:
            AGENT_REASONING = None
            r = lexsi_chat(messages, AGENT_MAX_TOKENS)
        if r.status_code != 200:
            raise RuntimeError(f"gateway returned {r.status_code}: {r.text[:200]}")
        body = r.json()
        if not body.get("choices"):
            raise RuntimeError(f"no answer from the gateway: {str(body)[:200]}")          # e.g. blocked by a guardrail
        msg = body["choices"][0].get("message") or {}
        text = msg.get("content") or ""
        if isinstance(text, list):
            text = "".join(b.get("text", "") for b in text if isinstance(b, dict))
        u = body.get("usage") or {}
        AGENT_CALLS.append({"agent": self.name, "seconds": round(time.time() - t0, 1), "input_tokens": u.get("prompt_tokens") or 0,
                            "output_tokens": u.get("completion_tokens") or 0, "model": f"{LEXSI_PROVIDER} · {LEXSI_MODEL}"})
        return text

    def _log(self, step, thought, tool, args, outcome, approved=None):
        AGENT_STEPS.append({"agent": self.name, "step": step, "thought": thought, "tool": tool,
                            "args": _compact(args, 300), "outcome": outcome[:300], "approved": approved})
        print(f"[{self.name} · {step}] {thought[:220]}\n      → {tool}({_compact(args, 160)}) → {outcome[:200]}")

    def run(self, task):
        messages = [{"role": "system", "content": self.system_prompt()}, {"role": "user", "content": task}]
        for step in range(1, self.max_steps + 1):
            text = self._llm(messages)
            act = parse_action(text)
            if act is None or (act["tool"] != "finish" and act["tool"] not in self.tools):
                why = "not a JSON action" if act is None else f"unknown tool '{act['tool']}'"
                self._log(step, (text or "")[:200], "—", {}, f"invalid reply ({why}); asked again")
                messages += [{"role": "assistant", "content": text or ""},
                             {"role": "user", "content": f"Invalid reply ({why}). Reply with exactly one JSON object as described in the AGENT PROTOCOL."}]
                continue
            thought, name, args = str(act["thought"]), act["tool"], act["args"] if isinstance(act["args"], dict) else {}
            if name == "finish":
                self.summary = str(args.get("summary", thought))
                self._log(step, thought, "finish", {}, self.summary)
                audit(self.name, "finish", self.summary[:300], "DONE")
                return self.summary
            tool = self.tools[name]
            approved = None
            if tool.approval:
                approved = human_gate(f"{self.name} wants {name}({_compact(args, 200)}). Its reasoning: {thought}", True)
            if approved is False:
                obs = {"rejected": True, "message": "The human reviewer rejected this action. Do not repeat it; choose another action or finish."}
            else:
                try:
                    obs = tool.fn(**args)
                except Exception as exc:
                    obs = {"error": f"{type(exc).__name__}: {str(exc)[:300]}"}
            out = _compact(obs)
            self._log(step, thought, name, args, out, approved)
            if tool.approval or name.startswith(("remove", "exclude", "promote")):
                audit(self.name, name, thought[:300], ("APPLIED" if approved else "REJECTED by reviewer") if approved is not None else "")
            messages += [{"role": "assistant", "content": json.dumps(act)}, {"role": "user", "content": "OBSERVATION\n" + out}]
        self.summary = f"stopped after {self.max_steps} steps"
        audit(self.name, "step limit", self.summary, "STOPPED")
        return self.summary


def run_llm_agent(agent, task, fallback):
    """Run an LLM agent; if the gateway fails, the rule-based version finishes the job and says so."""
    try:
        return agent.run(task)
    except Exception as exc:
        print(f"⚠️ {agent.name}: the LLM call failed ({str(exc)[:200]}) → the rule-based version takes over from here")
        audit(agent.name, "fallback", str(exc)[:200], "RULES")
        fallback()
        return "rule-based fallback"


def show_agent_trace(name):
    S = pd.DataFrame([s for s in AGENT_STEPS if s["agent"] == name])
    if not len(S):
        print(f"{name}: no LLM steps (AGENT_MODE = '{AGENT_MODE}')")
        return
    C_ = pd.DataFrame([c for c in AGENT_CALLS if c["agent"] == name])
    print(f"{name}: {len(S)} steps · {len(C_)} LLM calls · {C_.seconds.sum():.0f}s · {int(C_.input_tokens.sum()):,} input / "
          f"{int(C_.output_tokens.sum()):,} output tokens · {C_.model.iloc[0] if len(C_) else ''}")
    display(S[["step", "thought", "tool", "args", "outcome", "approved"]].style.hide(axis="index")
            .set_properties(**{"text-align": "left", "white-space": "pre-wrap", "font-size": "12px"}).set_caption(f"{name}: every step, in order"))



def agent_overview():
    tab = pd.DataFrame([
        ("Data-prep agent", "3.3", "which features to remove, and why",
         "profile, inspect_features, find_candidates, leakage_probe, remove_candidates, remove_features, restore_features",
         "data-owner exclusions and constant columns are removed; the leakage probe must end below its ceiling"),
        ("Retraining agent", "9.3", "whether a critical alert justifies retraining; whether the challenger is promoted",
         "read_job, retrain_challenger, verify_on_next_month, promote_challenger",
         "promotion only if the challenger beats the champion on the next unseen month by the margin"),
        ("Model-card writer", "8.1", "the model card, from the recorded facts", "fact sheet → card → automatic checks → one revision",
         "every number must match a recorded fact"),
    ], columns=["agent", "section", "decides", "tools", "rules that always hold"])
    tab["runs as"] = ["LLM" if AGENT_MODE == "llm" else "rules", "LLM" if AGENT_MODE == "llm" else "rules", WRITER]
    who = f"{LEXSI_PROVIDER} · {LEXSI_MODEL} through the Lexsi gateway" if AGENT_MODE == "llm" else "rule-based (no gateway)"
    display(tab.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).set_caption(f"Agents in this notebook · LLM: {who}"))

In [ ]:
AGENT_MODE = "llm" if WRITER == "lexsi" else "rules"   # "llm" = the agents reason with the LLM above · "rules" = the rule-based agents
AGENT_MAX_TOKENS = 3000                                 # per agent step (thinking tokens count against it)
agent_overview()

# 2 · Product options

## Problem statement
**How much credit should this applicant be offered?** A retail lender recommends a loan amount from the applicant's profile, bureau history and previous applications, to support limit setting and affordability checks.

## Training and test data
The notebook draws four disjoint sets from the source data, so nothing used to choose a model is used to grade it:

| Set | Rows (A100 default) | Used for |
|---|---|---|
| **Context (training)** | 5,000 | the rows a foundation model reads as context; training data for classic models |
| **Validation** | 2,000 | model selection, threshold choice, conformal calibration |
| **Test** | 3,000 | final, untouched performance report |
| **Production stream** | 6 × 1,000 | six later "months" for drift monitoring and retraining |

## Development vs production

| Choice | In development (this notebook) | In production (Lexsi platform) |
|---|---|---|
| Engine | TabTune open source on the notebook GPU | Lexsi-managed TabTune build with proprietary engines |
| Models | every model in the registry above | 17 foundation models plus classic AutoML |
| Adaptation | zero-shot, fine-tuning, LoRA | same, as managed training jobs |
| Scoring | batch scoring in the notebook | batch by tag, single case through the API |
| Governance | audit list in this notebook | registry, approvals, case audit trail, policies |

In [ ]:
# Use-case settings
TARGET_NAME = "credit amount"
N_CONTEXT, N_VALID, N_TEST, N_MONTH, MONTHS = (400, 200, 300, 150, 4) if FAST else (5000, 2000, 3000, 1000, 6)
ROSTER = ["TabICLv2", "TabPFN", "Mitra", "TabDPT"]                                   # foundation models raced zero-shot
if FAST:
    ROSTER = ROSTER[:2]
REQUIRE_COMMERCIAL = False                            # True = only models cleared for commercial use may win
RUN_PEFT = RUN_ENSEMBLE = (not FAST) or os.environ.get("TABTUNE_DEMO_PATHS") == "1"
TOP_K_FEATURES = 20 if FAST else 60                   # the preparation agent proposes keeping the strongest features
LORA_EPOCHS = 1 if FAST else 3
SEED = 7

# 3 · Data preparation

## 3.1 Load

In [ ]:
# @title Home Credit (processed parquet files on Google Drive) { display-mode: "form" }
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass
DATA_DIR = os.environ.get("HC_DATA_DIR", "/content/drive/MyDrive/datasets/home-credit-default-risk/processed")
data = pd.concat([pd.read_parquet(os.path.join(DATA_DIR, f)) for f in ("X_train.parquet", "X_val.parquet")], ignore_index=True)
data.columns = [re.sub(r"[^0-9A-Za-z_]+", "_", str(c)).strip("_") for c in data.columns]
data = data.loc[:, ~data.columns.duplicated()]
UID, TARGET_COL = "SK_ID_CURR", "AMT_CREDIT"
data[TARGET_COL] = pd.to_numeric(data[TARGET_COL], errors="coerce")
data = data[data[TARGET_COL].notna()].reset_index(drop=True)
# Fields that restate the answer or are only known after the loan exists
KNOWN_EXCLUDE = {c: "leaks the target (derived from or equal to the credit amount)" for c in data.columns
                 if c in ("AMT_GOODS_PRICE", "AMT_ANNUITY") or (c.startswith("FE_") and re.search("CREDIT|ANNUITY|GOODS", c))}
if "TARGET" in data:
    KNOWN_EXCLUDE["TARGET"] = "default outcome, observed only after the loan"
# Ratios defined from the credit amount itself (loan length = credit / annuity, payment rate = annuity / credit)
for c in ("FE_LOAN_LENGTH", "FE_PAYMENT_RATE"):
    if c in data:
        KNOWN_EXCLUDE[c] = "computed from the credit amount and the annuity"
CLASS_NAMES = None
GROUP_PREFIX, GROUP_LABEL = "CODE_GENDER", "gender"
print(f"{len(data):,} applications · {data.shape[1] - 2} candidate features · median credit {data[TARGET_COL].median():,.0f}")
display(data.sample(5, random_state=1).iloc[:, :12])

In [ ]:
# @title Four disjoint sets: context, validation, test and a production stream { display-mode: "form" }
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(SEED)
need = N_CONTEXT + N_VALID + N_TEST + N_MONTH * MONTHS
work = data.sample(min(len(data), max(need * 3, need)), random_state=SEED).reset_index(drop=True)
strat = work[TARGET_COL] if TASK == "classification" else pd.qcut(work[TARGET_COL].rank(method="first"), 10, labels=False)
dev, stream = train_test_split(work, train_size=N_CONTEXT + N_VALID + N_TEST, stratify=strat, random_state=SEED)
s2 = strat.loc[dev.index]
ctx, rest = train_test_split(dev, train_size=N_CONTEXT, stratify=s2, random_state=SEED)
val, test = train_test_split(rest, train_size=N_VALID, stratify=s2.loc[rest.index], random_state=SEED)
ctx, val, test, stream = (d.reset_index(drop=True) for d in (ctx, val, test, stream))
ALL_FEATURES = [c for c in data.columns if c not in (UID, TARGET_COL)]
sizes = pd.Series({"context (training)": len(ctx), "validation": len(val), "test": len(test), "production stream": len(stream)})
fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], subplot_titles=("Rows per set", f"Target in each set ({TARGET_NAME})"))
fig.add_trace(go.Bar(x=sizes.index, y=sizes.values, marker_color=[BLUE, TEAL, ORANGE, GREY], text=sizes.values, textposition="outside", showlegend=False), 1, 1)
for name, d, col in (("context", ctx, BLUE), ("validation", val, TEAL), ("test", test, ORANGE)):
    if TASK == "classification":
        fig.add_trace(go.Bar(x=[name], y=[d[TARGET_COL].mean() * 100], marker_color=col, showlegend=False, text=[f"{d[TARGET_COL].mean():.1%}"], textposition="outside"), 1, 2)
    else:
        fig.add_trace(go.Violin(y=d[TARGET_COL], name=name, line_color=col, box_visible=True, showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=420, title="Data split · stratified so every set has the same target mix")
fig.show()
record("data", source="Lending Club" if TASK == "classification" else "Home Credit", rows_available=len(data), context_rows=len(ctx),
       validation_rows=len(val), test_rows=len(test), candidate_features=len(ALL_FEATURES), target=TARGET_COL)

## 3.2 Data quality and distributions

In [ ]:
# @title Profile dashboard: types, missing values, cardinality, skew { display-mode: "form" }
NUM = [c for c in ALL_FEATURES if pd.api.types.is_numeric_dtype(ctx[c])]
CAT = [c for c in ALL_FEATURES if c not in NUM]
miss = ctx[ALL_FEATURES].isna().mean().mul(100)
uniq = ctx[ALL_FEATURES].nunique()
const = [c for c in ALL_FEATURES if uniq[c] <= 1]
binary = [c for c in NUM if ctx[c].dropna().isin([0, 1]).all() and c not in const]
cont = [c for c in NUM if c not in binary and uniq[c] > 20]
skew = ctx[cont].skew().abs().sort_values(ascending=False) if cont else pd.Series(dtype=float)

kinds = pd.Series({"continuous": len(cont), "binary / one-hot": len(binary), "low-cardinality numeric": len(NUM) - len(cont) - len(binary), "categorical": len(CAT)})
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.2, specs=[[{"type": "domain"}, {}, {}], [{}, {}, {}]],
                    subplot_titles=("Feature types", "Most incomplete features (% missing)", "Distinct values per feature (log)",
                                    "Most skewed features", "Heaviest-tailed feature · raw", "Same feature · signed log"))
fig.add_trace(go.Pie(labels=kinds.index, values=kinds.values, hole=0.55, marker_colors=[BLUE, TEAL, GOLD, ORANGE], showlegend=False, textinfo="label+value"), 1, 1)
tm = miss.sort_values(ascending=False).head(12)[::-1]
fig.add_trace(go.Bar(x=tm.values, y=[c[:26] for c in tm.index], orientation="h", marker_color=[RED if v > 50 else GREY for v in tm.values], showlegend=False), 1, 2)
fig.add_trace(go.Histogram(x=np.log10(uniq.clip(lower=1)), nbinsx=30, marker_color=BLUE, showlegend=False), 1, 3)
if len(skew):
    sk = skew.head(10)[::-1]; f0 = skew.index[0]
    fig.add_trace(go.Bar(x=sk.values, y=[c[:26] for c in sk.index], orientation="h", marker_color=ORANGE, showlegend=False), 2, 1)
    fig.add_trace(go.Histogram(x=ctx[f0], nbinsx=60, marker_color=GREY, showlegend=False), 2, 2)
    fig.add_trace(go.Histogram(x=np.sign(ctx[f0]) * np.log1p(ctx[f0].abs()), nbinsx=60, marker_color=TEAL, showlegend=False), 2, 3)
fig.update_layout(template="plotly_white", height=720, title="Data profile of the context set")
fig.show()
record("data", features_with_missing=int((miss > 0).sum()), constant_features=len(const), categorical_features=len(CAT),
       heavily_skewed_features=int((skew > 2).sum()))

In [ ]:
# @title Signal dashboard: which features move the target, and which repeat each other { display-mode: "form" }
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

def encode(df, cols=None):
    """Numeric view of a frame: categoricals become stable integer codes, missing values stay missing."""
    cols = cols or ALL_FEATURES
    out = df[cols].copy()
    for c in cols:
        if c in CAT_CODES:
            out[c] = out[c].map(CAT_CODES[c]).astype(float)
    return out.replace([np.inf, -np.inf], np.nan)

from sklearn import metrics as skm
PRIMARY = "roc_auc" if TASK == "classification" else "r2"

def score_of(model, X):
    """Probability of the positive class, or the predicted value."""
    if TASK == "classification":
        p = np.asarray(model.predict_proba(X))
        return p[:, 1] if p.ndim == 2 else p
    return np.asarray(model.predict(X), dtype=float)

def metric(y, s):
    return float(skm.roc_auc_score(y, s) if TASK == "classification" else skm.r2_score(y, s))

CAT_CODES = {c: {v: i for i, v in enumerate(sorted(data[c].dropna().astype(str).unique()))} for c in CAT}
for c in CAT:
    for d in (ctx, val, test, stream):
        d[c] = d[c].astype(str).where(d[c].notna(), None)
Xc_num = encode(ctx)
fill = Xc_num.median()
mi_fn = mutual_info_classif if TASK == "classification" else mutual_info_regression
MI = pd.Series(mi_fn(Xc_num.fillna(fill).fillna(0), ctx[TARGET_COL], random_state=0), index=ALL_FEATURES).sort_values(ascending=False)
top = MI.head(16).index.tolist()
corr = Xc_num[top].corr()
tcorr = Xc_num.corrwith(ctx[TARGET_COL].astype(float)).abs()

fig = make_subplots(rows=1, cols=2, column_widths=[0.38, 0.62], subplot_titles=("Mutual information with the target (top 16)", "Correlation among those features"))
fig.add_trace(go.Bar(x=MI.head(16).values[::-1], y=[c[:28] for c in top[::-1]], orientation="h", marker_color=BLUE, showlegend=False), 1, 1)
fig.add_trace(go.Heatmap(z=corr.values, x=[c[:16] for c in top], y=[c[:16] for c in top], colorscale="RdBu_r", zmin=-1, zmax=1), 1, 2)
fig.update_layout(template="plotly_white", height=600, title="Signal and redundancy")
fig.show()

fig = make_subplots(rows=1, cols=4, subplot_titles=[c[:26] for c in top[:4]])
for j, c in enumerate(top[:4], 1):
    b = pd.qcut(Xc_num[c].rank(method="first"), 10, labels=False)
    g = pd.DataFrame({"b": b, "x": Xc_num[c], "y": ctx[TARGET_COL]}).groupby("b").agg(x=("x", "median"), y=("y", "mean"))
    fig.add_trace(go.Scatter(x=g.x, y=g.y * (100 if TASK == "classification" else 1), mode="lines+markers", line=dict(color=ORANGE, width=3), showlegend=False), 1, j)
fig.update_yaxes(title_text="default rate (%)" if TASK == "classification" else f"mean {TARGET_COL}", row=1, col=1)
fig.update_layout(template="plotly_white", height=360, title="How the target responds across deciles of the four strongest features")
fig.show()

## 3.3 Data preparation agent (LLM)

The agent's tools are the checks a careful analyst would make. The LLM decides which checks to run, reads the evidence, and proposes removals; a human approves each one; rules check the result.

| Tool | What it returns or does |
|---|---|
| `profile()` | rows, feature types, missing values, constant columns, data-owner exclusions, the strongest features |
| `inspect_features(features)` | per feature: type, missing %, distinct values, signal rank, mutual information, correlation with the target |
| `find_candidates(check)` | candidates for one check (data-owner exclusions, constant, mostly missing, identifier-like, target leakage, redundant twins, weak signal), with evidence |
| `leakage_probe()` | **too good to be true**: a quick probe model's score against a ceiling no real lending model reaches, the features it leans on, and what removing each does |
| `remove_candidates(check, keep, reason)` · `remove_features(features, category, reason)` | removes features *(human approval)* |
| `restore_features(features, reason)` | undoes a removal *(human approval)* |

The agent only sees the context (training) set, never validation or test rows. Set `AGENT_MODE = "rules"` in section 1 to run the rule-based agent instead.

In [ ]:
# @title The data preparation agent (definition) { display-mode: "form" }
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split as _tts

LEAK_CEILING = 0.97 if TASK == "classification" else 0.85


class DataPrepAgent:
    def __init__(self, frame, features):
        self.frame, self.features, self.steps, self.dropped, self.probe_log = frame, list(features), [("candidate features", len(features))], {}, []

    def _propose(self, name, cols, reason, recommended=True):
        cols = [c for c in cols if c in self.features]
        if not cols:
            audit("prep-agent", name, "nothing to do")
            return
        audit("prep-agent", name, f"{len(cols)} feature(s): {', '.join(cols[:5])}{'…' if len(cols) > 5 else ''} — {reason}")
        if human_gate(f"{name}: remove {len(cols)} feature(s)?", recommended):
            self.features = [c for c in self.features if c not in cols]
            self.dropped.update({c: name for c in cols})
        self.steps.append((name, len(self.features)))

    def _probe(self, importance=True):
        """Fit a quick gradient-boosting probe on the current features; return its holdout score and the feature it relies on most."""
        X = encode(self.frame, self.features); y = self.frame[TARGET_COL]
        n = min(len(X), 4000)
        X, y = X.iloc[:n], y.iloc[:n]
        strat = y if TASK == "classification" else None
        Xa, Xb, ya, yb = _tts(X, y, test_size=0.3, random_state=0, stratify=strat)
        Probe_ = HistGradientBoostingClassifier if TASK == "classification" else HistGradientBoostingRegressor
        m = Probe_(max_iter=200, random_state=0).fit(Xa, ya)
        s = metric(yb, m.predict_proba(Xb)[:, 1] if TASK == "classification" else m.predict(Xb))
        if not importance:
            return s, None
        imp = permutation_importance(m, Xb, yb, n_repeats=2, random_state=0, scoring="roc_auc" if TASK == "classification" else "r2")
        imp = pd.Series(imp.importances_mean, index=Xb.columns).sort_values(ascending=False)
        return s, imp

    def _drop_score(self, feature):
        keep = self.features
        self.features = [f for f in keep if f != feature]
        try:
            return self._probe(importance=False)[0]
        finally:
            self.features = keep

    def leakage_probe(self, max_rounds=8):
        for r in range(max_rounds):
            score, imp = self._probe()
            if score > LEAK_CEILING:
                # leaks often need two fields together, so test removing each top suspect; prefer an engineered field whose removal
                # does real damage (a raw application field such as income is rarely the culprit on its own)
                suspects = imp.head(4).index.tolist()
                after = {f: self._drop_score(f) for f in suspects}
                best_drop = score - min(after.values())
                engineered = [f for f in sorted(suspects, key=after.get) if f.lower().startswith("fe_") and score - after[f] >= 0.25 * best_drop]
                top = engineered[0] if engineered else min(suspects, key=after.get)
            else:
                top = imp.index[0]
            share = float(imp[top] / max(imp.clip(lower=0).sum(), 1e-9))
            self.probe_log.append({"round": r + 1, "probe_score": score, "top_feature": top, "importance_share": share})
            if score <= LEAK_CEILING:
                audit("prep-agent", "too good to be true", f"probe {PRIMARY} {score:.3f} ≤ ceiling {LEAK_CEILING}", "believable: stop")
                break
            n_before = len(self.features)
            self._propose("too good to be true", [top], f"probe {PRIMARY} {score:.3f} is above {LEAK_CEILING}; the probe leans on this feature "
                          f"for {share:.0%} of its accuracy, and removing it breaks the probe's shortcut, so it probably restates the answer")
            if len(self.features) == n_before:
                break
        return self

    def run(self):
        f = self.frame
        self._propose("known exclusions", list(KNOWN_EXCLUDE), "identifiers or fields flagged by the data owner")
        self._propose("constant columns", const, "one value only, no information")
        self._propose("mostly missing", miss[miss > 80].index.tolist(), "more than 80% missing")
        ident = [c for c in self.features if c in CAT and uniq[c] > 0.5 * len(f)]
        self._propose("identifier-like text", ident, "almost one distinct value per row")
        leak = tcorr[(tcorr > 0.95)].index.tolist() if TASK == "regression" else tcorr[tcorr > 0.98].index.tolist()
        self._propose("target leakage", leak, "near-perfect correlation with the target; it would not exist at decision time")
        self.leakage_probe()
        strong = [c for c in MI.index if c in self.features and c in NUM][:150]
        cm = Xc_num[strong].corr().abs()
        twins = []
        for i, a in enumerate(strong):
            if a in twins:
                continue
            twins += [b for b in strong[i + 1:] if b not in twins and cm.loc[a, b] > 0.97]
        self._propose("redundant twins", twins, "correlated above 0.97 with a stronger feature")
        weak = [c for c in MI.index if c in self.features][TOP_K_FEATURES:]
        self._propose(f"keep the {TOP_K_FEATURES} strongest", weak, "little or no signal; fewer features means faster scoring and clearer explanations")
        return self

In [ ]:
# @title LLM data-preparation agent: tools and guardrails { display-mode: "form" }
CATEGORY = {"known_exclusions": "known exclusions", "constant": "constant columns", "mostly_missing": "mostly missing",
            "identifier_like": "identifier-like text", "target_leakage": "target leakage", "redundant_twins": "redundant twins",
            "weak_signal": f"keep the {TOP_K_FEATURES} strongest"}


class LLMDataPrepAgent(DataPrepAgent):
    """The rule agent's checks, offered to an LLM as tools. The LLM decides; a human approves removals; rules guard the result."""

    def __init__(self, frame, features):
        super().__init__(frame, features)
        self.reasons = {}

    # ---- evidence ------------------------------------------------------------------------------------------
    def _rank(self, c):
        return int(MI.index.get_loc(c)) + 1 if c in MI.index else None

    def candidates(self, check):
        f = self.features
        if check == "known_exclusions":
            return [(c, KNOWN_EXCLUDE[c]) for c in KNOWN_EXCLUDE if c in f]
        if check == "constant":
            return [(c, "one value only") for c in const if c in f]
        if check == "mostly_missing":
            return [(c, f"{miss[c]:.0f}% missing") for c in miss[miss > 80].index if c in f]
        if check == "identifier_like":
            return [(c, f"{uniq[c]:,} distinct values in {len(self.frame):,} rows") for c in f if c in CAT and uniq[c] > 0.5 * len(self.frame)]
        if check == "target_leakage":
            lim = 0.95 if TASK == "regression" else 0.98
            return [(c, f"|correlation with target| = {tcorr[c]:.3f}") for c in tcorr[tcorr > lim].index if c in f]
        if check == "redundant_twins":
            strong = [c for c in MI.index if c in f and c in NUM][:150]
            cm = Xc_num[strong].corr().abs(); out, seen = [], set()
            for i, a in enumerate(strong):
                if a in seen:
                    continue
                for b in strong[i + 1:]:
                    if b not in seen and cm.loc[a, b] > 0.97:
                        seen.add(b); out.append((b, f"correlation {cm.loc[a, b]:.3f} with stronger feature {a}"))
            return out
        if check == "weak_signal":
            return [(c, f"signal rank {self._rank(c)}, mutual information {MI[c]:.4f}") for c in [c for c in MI.index if c in f][TOP_K_FEATURES:]]
        raise ValueError(f"unknown check '{check}'; use one of {list(CATEGORY)}")

    def t_profile(self):
        f = self.features
        return {"task": TASK, "target": TARGET_COL, "context_rows": len(self.frame), "features_now": len(f),
                "numeric": sum(c in NUM for c in f), "categorical": sum(c in CAT for c in f),
                "features_with_missing": int((miss[f] > 0).sum()), "constant_columns": sum(c in const for c in f),
                "data_owner_exclusions": {c: v for c, v in KNOWN_EXCLUDE.items() if c in f},
                "strongest_features": [{"feature": c, "mutual_information": round(float(MI[c]), 4)} for c in [c for c in MI.index if c in f][:12]],
                "leak_ceiling": LEAK_CEILING, "suggested_size": TOP_K_FEATURES, "checks_available": list(CATEGORY)}

    def t_inspect(self, features):
        rows = []
        for c in features[:30]:
            if c not in self.frame:
                rows.append({"feature": c, "error": "not in the data"}); continue
            rows.append({"feature": c, "type": "categorical" if c in CAT else "numeric", "missing_pct": round(float(miss.get(c, 0)), 1),
                         "distinct": int(uniq.get(c, self.frame[c].nunique())), "signal_rank": self._rank(c),
                         "mutual_information": round(float(MI[c]), 4) if c in MI.index else None,
                         "abs_corr_with_target": round(float(tcorr[c]), 3) if c in tcorr.index and pd.notna(tcorr[c]) else None,
                         "currently_in_model": c in self.features})
        return rows

    def t_find(self, check):
        found = self.candidates(check)
        return {"check": check, "category": CATEGORY[check], "count": len(found),
                "candidates": [{"feature": c, "evidence": e} for c, e in found[:40]]}

    def t_probe(self):
        score, imp = self._probe()
        suspects = imp.head(4).index.tolist()
        after = {s: round(float(self._drop_score(s)), 4) for s in suspects} if score > LEAK_CEILING else {}
        if after:                                   # same preference as the rule agent: an engineered field whose removal breaks the shortcut
            best_drop = score - min(after.values())
            eng = [f for f in sorted(after, key=after.get) if f.lower().startswith("fe_") and score - after[f] >= 0.25 * best_drop]
            top = eng[0] if eng else min(after, key=after.get)
        else:
            top = imp.index[0]
        share = float(imp[top] / max(imp.clip(lower=0).sum(), 1e-9))
        self.probe_log.append({"round": len(self.probe_log) + 1, "probe_score": score, "top_feature": top, "importance_share": share})
        return {"probe_metric": PRIMARY, "probe_score": round(float(score), 4), "ceiling": LEAK_CEILING, "above_ceiling": bool(score > LEAK_CEILING),
                "features_the_probe_leans_on": [{"feature": c, "importance_share": round(float(v / max(imp.clip(lower=0).sum(), 1e-9)), 3),
                                                 "engineered": c.lower().startswith("fe_")} for c, v in imp.head(6).items()],
                "score_after_removing_each_suspect": after, "suggested": top if score > LEAK_CEILING else None}

    # ---- actions (the human gate runs before these) ---------------------------------------------------------
    def _remove(self, cols, category, reason):
        cols = [c for c in cols if c in self.features and c not in (UID, TARGET_COL)]
        self.features = [c for c in self.features if c not in cols]
        for c in cols:
            self.dropped[c], self.reasons[c] = category, reason
        if cols:
            self.steps.append((category, len(self.features)))
        return {"removed": cols, "features_now": len(self.features)}

    def t_remove_candidates(self, check, keep=None, reason=""):
        cols = [c for c, _ in self.candidates(check) if c not in set(keep or [])]
        return self._remove(cols, CATEGORY[check], reason or CATEGORY[check])

    def t_remove(self, features, category="other", reason=""):
        unknown = [c for c in features if c not in self.features]
        out = self._remove(features, category, reason)
        if unknown:
            out["not_in_current_features"] = unknown
        return out

    def t_restore(self, features, reason=""):
        back = [c for c in features if c in self.dropped and c in ALL_FEATURES]
        self.features += [c for c in back if c not in self.features]
        for c in back:
            self.dropped.pop(c, None); self.reasons.pop(c, None)
        self.steps.append(("restored", len(self.features)))
        return {"restored": back, "features_now": len(self.features)}

    # ---- guardrails ------------------------------------------------------------------------------------------
    def enforce(self):
        for check in ("known_exclusions", "constant"):
            left = [c for c, _ in self.candidates(check)]
            if left:
                audit("rules", "guardrail", f"{CATEGORY[check]} still present: {', '.join(left[:5])}", "REMOVED by policy")
                self._remove(left, CATEGORY[check], "removed by policy after the agent finished")
        score, _ = self._probe(importance=False)
        if score > LEAK_CEILING:
            audit("rules", "guardrail", f"probe {PRIMARY} {score:.3f} still above {LEAK_CEILING}", "rule-based leakage probe runs")
            self.leakage_probe()
        else:
            audit("rules", "guardrail", f"probe {PRIMARY} {score:.3f} ≤ ceiling {LEAK_CEILING}; data-owner exclusions and constant columns removed", "OK")

    def run(self):
        if AGENT_MODE != "llm":
            return super().run()
        llm = LLMAgent(
            "data-prep agent",
            role=f"You prepare the feature set for a bank's {TASK} model of '{TARGET_COL}' ({TARGET_NAME}). You see only the training (context) rows.",
            goal=("Leave a feature set that is clean, has no leakage and keeps the real signal. Look at the profile, run the checks you judge "
                  "necessary (consider every check in find_candidates), remove what the evidence justifies, and run the leakage probe after the "
                  "obvious removals and again after any removal it prompts, until its score is below the ceiling. Aim for about "
                  f"{TOP_K_FEATURES} features unless the evidence argues otherwise."),
            rules=["Remove data-owner exclusions and constant columns; they are mandatory.",
                   "A feature computed from the target, or only known after the decision, is leakage even if its correlation looks modest.",
                   "Prefer removing an engineered feature (name starting fe_/FE_) over a raw application field when both explain a leak.",
                   "Do not remove a feature only because its name sounds risky; use the evidence."],
            tools=[Tool("profile", self.t_profile, "overview of the data and the current features"),
                   Tool("inspect_features", self.t_inspect, "evidence for specific features", {"features": "list of feature names"}),
                   Tool("find_candidates", self.t_find, "candidates for one check, with evidence", {"check": " | ".join(CATEGORY)}),
                   Tool("leakage_probe", self.t_probe, "too-good-to-be-true probe on the current features"),
                   Tool("remove_candidates", self.t_remove_candidates, "remove every candidate of a check except those listed in keep",
                        {"check": "a check name", "keep": "list of features to keep (optional)", "reason": "why"}, approval=True),
                   Tool("remove_features", self.t_remove, "remove specific features",
                        {"features": "list", "category": " | ".join(CATEGORY.values()) + " | too good to be true | other", "reason": "why"}, approval=True),
                   Tool("restore_features", self.t_restore, "undo a removal", {"features": "list", "reason": "why"}, approval=True)],
            max_steps=26)
        self.llm = llm
        run_llm_agent(llm, f"Start. {len(self.features)} candidate features for target '{TARGET_COL}'.", lambda: DataPrepAgent.run(self))
        self.enforce()
        return self

In [ ]:
prep_agent = (LLMDataPrepAgent if AGENT_MODE == "llm" else DataPrepAgent)(ctx, ALL_FEATURES)
prep_agent.run()                                   # LLM decides → human approves → rules check the result
FEATURES = prep_agent.features

In [ ]:
show_agent_trace("data-prep agent")                # every step: the agent's reasoning, the tool it called, what came back

In [ ]:
# @title What the agent did { display-mode: "form" }
steps = pd.DataFrame(prep_agent.steps, columns=["step", "features left"])
why = pd.Series(prep_agent.dropped).value_counts()
probe = pd.DataFrame(prep_agent.probe_log)
fig = make_subplots(rows=1, cols=3, column_widths=[0.42, 0.28, 0.3], specs=[[{}, {"type": "domain"}, {}]],
                    subplot_titles=("Features remaining after each step", "Why features were removed", f"'Too good to be true' probe ({PRIMARY})"))
fig.add_trace(go.Funnel(y=steps.step, x=steps["features left"], marker_color=BLUE, textinfo="value", showlegend=False), 1, 1)
fig.add_trace(go.Pie(labels=why.index, values=why.values, hole=0.5, marker_colors=PALETTE, showlegend=False, textinfo="label+value"), 1, 2)
if len(probe):
    fig.add_trace(go.Scatter(x=probe["round"], y=probe.probe_score, mode="lines+markers+text", text=probe.top_feature.str[:18], textposition="top center",
                             line=dict(color=ORANGE, width=3), showlegend=False), 1, 3)
    fig.add_trace(go.Scatter(x=[probe["round"].min() - 0.3, probe["round"].max() + 0.3], y=[LEAK_CEILING] * 2, mode="lines",
                             line=dict(color=RED, dash="dot"), showlegend=False, hoverinfo="skip"), 1, 3)
    fig.update_xaxes(title_text="round (label = feature the probe relied on most)", dtick=1, row=1, col=3)
fig.update_layout(template="plotly_white", height=520, title=f"Data preparation agent · {len(ALL_FEATURES)} → {len(FEATURES)} features")
fig.show()
leak_steps = ("known exclusions", "target leakage", "too good to be true")
record("preparation", features_before=len(ALL_FEATURES), features_after_agent=len(FEATURES), features_removed=len(prep_agent.dropped),
       leakage_features_removed=int(pd.Series(prep_agent.dropped).isin(leak_steps).sum()),
       probe_score_first=float(probe.probe_score.iloc[0]) if len(probe) else float("nan"),
       probe_score_final=float(probe.probe_score.iloc[-1]) if len(probe) else float("nan"), leak_ceiling=LEAK_CEILING)

## 3.4 Automated feature engineering

In [ ]:
# @title Generate candidate features, keep the ones a probe model says help { display-mode: "form" }
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

base = [c for c in MI.index if c in FEATURES and c in cont][:8]
recipes = []
for i, a in enumerate(base):
    for b in base[i + 1:]:
        recipes += [(f"fe_prod__{a}__{b}", "prod", a, b), (f"fe_diff__{a}__{b}", "diff", a, b)]
        if (ctx[b] > 0).all():
            recipes.append((f"fe_ratio__{a}__{b}", "ratio", a, b))

def apply_recipes(df, recs):
    out = {}
    for name, op, a, b in recs:
        out[name] = df[a] * df[b] if op == "prod" else df[a] - df[b] if op == "diff" else df[a] / df[b].where(df[b] > 0)
    return pd.DataFrame(out, index=df.index).replace([np.inf, -np.inf], np.nan)

cand = apply_recipes(ctx, recipes)
cand_mi = pd.Series(mi_fn(cand.fillna(cand.median()).fillna(0), ctx[TARGET_COL], random_state=0), index=cand.columns) if len(recipes) else pd.Series(dtype=float)
parent = pd.Series({n: max(MI[a], MI[b]) for n, _, a, b in recipes})
gain = (cand_mi - parent).sort_values(ascending=False)
KEPT = gain.head(8).index.tolist()
Probe = HistGradientBoostingRegressor if TASK == "regression" else HistGradientBoostingClassifier
scoring = "r2" if TASK == "regression" else "roc_auc"
cv = lambda X: cross_val_score(Probe(max_iter=120, random_state=0), X, ctx[TARGET_COL], cv=3, scoring=scoring).mean()
base_X = encode(ctx, FEATURES)
score_raw, score_fe = cv(base_X), (cv(pd.concat([base_X, cand[KEPT]], axis=1)) if KEPT else np.nan)
helps = bool(KEPT) and score_fe > score_raw + 0.0005
audit("prep-agent", "feature engineering", f"probe {scoring} {score_raw:.4f} → {score_fe:.4f} with {len(KEPT)} engineered features",
      "worth adding" if helps else "no gain: keep raw features")
ADD_ENGINEERED = helps and human_gate(f"Add {len(KEPT)} engineered features?", True)
KEPT_RECIPES = [r for r in recipes if r[0] in KEPT] if ADD_ENGINEERED else []
if ADD_ENGINEERED:
    for d in (ctx, val, test, stream):
        fe = apply_recipes(d, KEPT_RECIPES)
        for c in fe.columns:
            d[c] = fe[c]
    FEATURES = FEATURES + KEPT
    MI = pd.concat([MI, cand_mi[KEPT]]).sort_values(ascending=False)

fig = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.4, 0.24],
                    subplot_titles=(f"{len(recipes)} candidates: signal vs best parent", "Best candidates: gain over parents", f"Probe model ({scoring}, 3-fold)"))
if len(recipes):
    fig.add_trace(go.Scatter(x=parent, y=cand_mi, mode="markers", showlegend=False,
                             marker=dict(color=[ORANGE if n in KEPT else GREY for n in cand_mi.index], size=[10 if n in KEPT else 5 for n in cand_mi.index])), 1, 1)
    g = gain[KEPT][::-1]
    fig.add_trace(go.Bar(x=g.values, y=[n.replace("fe_", "").replace("__", " · ")[:40] for n in g.index], orientation="h", marker_color=ORANGE, showlegend=False), 1, 2)
fig.add_trace(go.Bar(x=["raw", "+ engineered"], y=[score_raw, score_fe], marker_color=[GREY, ORANGE], text=[f"{score_raw:.4f}", f"{score_fe:.4f}"], textposition="outside", showlegend=False), 1, 3)
fig.update_yaxes(range=[np.nanmin([score_raw, score_fe]) * 0.97, np.nanmax([score_raw, score_fe]) * 1.02], row=1, col=3)
fig.update_layout(template="plotly_white", height=520, title="Automated feature engineering · " + ("added to the feature set" if ADD_ENGINEERED else "not added: no measurable gain"))
fig.show()
record("features", final_feature_count=len(FEATURES), engineered_features_added=len(KEPT_RECIPES), strongest_features=", ".join([c for c in MI.index if c in FEATURES][:5]))

In [ ]:
# @title The prepared data in two dimensions, and the model-ready frames { display-mode: "form" }
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
Xtr, ytr = ctx[FEATURES], ctx[TARGET_COL]
Xva, yva = val[FEATURES], val[TARGET_COL]
Xte, yte = test[FEATURES], test[TARGET_COL]
E = encode(ctx, FEATURES); FILL = E.median()
Z = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(E.fillna(FILL).fillna(0)))
fig = go.Figure()
if TASK == "classification":
    for cls, col in ((0, BLUE), (1, RED)):
        m = (ytr == cls).to_numpy()
        fig.add_trace(go.Scattergl(x=Z[m, 0], y=Z[m, 1], mode="markers", name=CLASS_NAMES[cls], marker=dict(size=4, color=col, opacity=0.45)))
else:
    fig.add_trace(go.Scattergl(x=Z[:, 0], y=Z[:, 1], mode="markers", marker=dict(size=4, color=np.log1p(ytr), colorscale="Viridis", opacity=0.6, colorbar=dict(title="log target"))))
fig.update_layout(template="plotly_white", height=480, title="Context set after preparation (PCA projection)")
fig.show()
display(Markdown(f"""
### Data preparation: conclusions
- **{len(ALL_FEATURES)} candidate features became {len(FEATURES)}** after the agent's approved steps; {FACTS['preparation']['leakage_features_removed']} were removed as leakage or known exclusions.
- **{FACTS['data']['features_with_missing']} features have missing values** and {FACTS['data']['heavily_skewed_features']} are heavily skewed; the foundation models apply their own robust scaling and missing-value handling, so no manual imputation was needed.
- **Engineered features were {'added' if ADD_ENGINEERED else 'not added'}** (probe {scoring} {score_raw:.4f} → {score_fe:.4f}).
- Every step is in the audit trail at the end of the notebook.
"""))

## 3.5 The feature list
Every candidate feature, with what happened to it: kept as a model input or removed (by which step, and the agent's reason). Engineered features added in 3.4 are included. Use the filter to switch between the model inputs, the removed features, or all of them.

In [ ]:
# @title Feature list: every candidate feature and what happened to it { display-mode: "form" }
def _kind(c):
    if c.startswith("fe_") and c not in ALL_FEATURES:
        return "engineered (3.4)"
    return "categorical" if c in CAT else "binary / one-hot" if c in binary else "continuous" if c in cont else "numeric (few values)"

why_ = getattr(prep_agent, "reasons", {})
rows_ = []
for c in list(dict.fromkeys(ALL_FEATURES + FEATURES)):
    col_ = ctx[c] if c in ctx else None
    corr_ = tcorr.get(c) if c in tcorr.index else (encode(ctx, [c])[c].corr(ctx[TARGET_COL].astype(float)) if c in ctx and c not in CAT else np.nan)
    rows_.append({"feature": c, "kind": _kind(c), "status": "model input" if c in FEATURES else "removed",
                  "removed by": prep_agent.dropped.get(c, ""), "reason": why_.get(c, KNOWN_EXCLUDE.get(c, "")) if c not in FEATURES else "",
                  "signal rank": int(MI.index.get_loc(c)) + 1 if c in MI.index else None,
                  "mutual information": float(MI[c]) if c in MI.index else np.nan, "|corr| with target": abs(float(corr_)) if pd.notna(corr_) else np.nan,
                  "missing %": float(col_.isna().mean() * 100) if col_ is not None else np.nan, "distinct values": int(col_.nunique()) if col_ is not None else None})
FEATURE_TABLE = pd.DataFrame(rows_).sort_values(["status", "signal rank"], ascending=[True, True], na_position="last").reset_index(drop=True)

inp, rem = FEATURE_TABLE[FEATURE_TABLE.status == "model input"], FEATURE_TABLE[FEATURE_TABLE.status == "removed"]
fig = make_subplots(rows=1, cols=3, column_widths=[0.42, 0.29, 0.29], specs=[[{}, {"type": "domain"}, {"type": "domain"}]],
                    subplot_titles=(f"The {len(inp)} model inputs, by signal (mutual information)", "Model inputs by kind", f"{len(rem)} removed, by step"))
top_ = inp.dropna(subset=["mutual information"]).head(25)[::-1]
fig.add_trace(go.Bar(x=top_["mutual information"], y=top_.feature.str[:30], orientation="h", showlegend=False,
                     marker_color=[ORANGE if k.startswith("engineered") else BLUE for k in top_.kind]), 1, 1)
k_, r_ = inp.kind.value_counts(), rem["removed by"].value_counts()
fig.add_trace(go.Pie(labels=k_.index, values=k_.values, hole=0.5, marker_colors=PALETTE, textinfo="label+value", showlegend=False), 1, 2)
if len(r_):
    fig.add_trace(go.Pie(labels=r_.index, values=r_.values, hole=0.5, marker_colors=PALETTE[2:] + PALETTE[:2], textinfo="label+value", showlegend=False), 1, 3)
fig.update_layout(template="plotly_white", height=max(520, 22 * len(top_) + 140), title=f"Feature list · {len(ALL_FEATURES)} candidates → {len(FEATURES)} model inputs (orange = engineered)")
fig.show()


def explore_features(show="model inputs", search=""):
    t = FEATURE_TABLE if show == "all" else FEATURE_TABLE[FEATURE_TABLE.status == ("model input" if show == "model inputs" else "removed")]
    if search:
        t = t[t.feature.str.contains(search, case=False, regex=False)]
    display(t.style.hide(axis="index").format({"mutual information": "{:.4f}", "|corr| with target": "{:.3f}", "missing %": "{:.1f}"}, na_rep="—")
            .set_properties(**{"text-align": "left", "white-space": "pre-wrap", "font-size": "12px"}).set_caption(f"{len(t)} features"))

In [ ]:
W.interact(explore_features, show=W.Dropdown(options=["model inputs", "removed", "all"], value="model inputs"), search=W.Text(value="", placeholder="filter by name"));

# 4 · Model training

## 4.1 One foundation model, step by step
Before racing many models, here is the whole TabTune workflow for one: create the pipeline, fit it, predict, evaluate. The model runs **zero-shot**: `fit` hands it the context rows, and it predicts from them without any gradient training.

In [ ]:
pipe = TabularPipeline(model_name="TabPFN", task_type=TASK, tuning_strategy="inference", tuning_params={"device": DEVICE})

In [ ]:
pipe.fit(Xtr, ytr)

In [ ]:
p_val = pipe.predict(Xva)                      # predicted credit amount, one per application
p_val[:5]

In [ ]:
val_report = pipe.evaluate(Xva, yva)

## 4.2 AutoML race
Every candidate is fitted on the **context** set and scored on the **validation** set: zero-shot foundation models against classic models. The helper cell defines `enter()`, which fits one candidate, scores it and adds it to the leaderboard. (`TabularLeaderboard(...).add_all().run()` does the same in one line; this version keeps every fitted model for the later sections.)

In [ ]:
# @title Race helpers: fit a candidate, score it on validation, log it { display-mode: "form" }
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb

class Tabular:                                         # classic models see the numeric encoding
    def __init__(self, est): self.est = est
    def fit(self, X, y): self.est.fit(encode(X, list(X.columns)), y); return self
    def predict(self, X): return self.est.predict(encode(X, list(X.columns)))
    def predict_proba(self, X): return self.est.predict_proba(encode(X, list(X.columns)))

def fit_tfm(name, X, y, strategy="inference", tuning_params=None, **kw):
    return TabularPipeline(model_name=name, task_type=TASK, tuning_strategy=strategy,
                           tuning_params={"device": DEVICE, **(tuning_params or {})}, **kw).fit(X, y)

RACE, FITTED, VAL_SCORES = [], {}, {}
def enter(name, family, build, tuning="none"):
    t0 = time.time()
    try:
        model = build()
        fit_s = time.time() - t0
        t1 = time.time(); s = score_of(model, Xva); pred_s = time.time() - t1
        FITTED[name], VAL_SCORES[name] = model, s
        spec = TABTUNE_MODELS.get(name, {})
        RACE.append({"model": name, "family": family, "tuning": tuning, PRIMARY: metric(yva, s), "fit_seconds": fit_s,
                     "ms_per_row": pred_s / len(Xva) * 1000, "commercial_use": spec.get("commercial_use", "yes"), "status": "ok"})
        print(f"✅ {name:22} {PRIMARY} {RACE[-1][PRIMARY]:.4f} · fit {fit_s:.1f}s")
    except Exception as exc:
        RACE.append({"model": name, "family": family, "tuning": tuning, "status": f"failed: {str(exc)[:90]}"})
        print(f"❌ {name:22} {str(exc)[:110]}")

Probe = HistGradientBoostingRegressor if TASK == "regression" else HistGradientBoostingClassifier
Linear = (lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=500))) if TASK == "classification" \
         else (lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge()))

In [ ]:
for name in ROSTER:                                                     # zero-shot foundation models
    enter(name, "Foundation model (zero-shot)", lambda n=name: fit_tfm(n, Xtr, ytr), "none (zero-shot)")

enter("Linear baseline", "Classic ML", lambda: Tabular(Linear()).fit(Xtr, ytr))
enter("HistGradientBoosting", "Classic ML", lambda: Tabular(Probe(max_iter=300, random_state=0)).fit(Xtr, ytr), "defaults")

## 4.3 Hyperparameter tuning
Classic models need a search: XGBoost gets a randomised search with 3-fold cross-validation on the context set. Foundation models need no search for zero-shot use; their tuning lever is **adaptation**, tested next with LoRA fine-tuning and an ensemble.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
XGB = xgb.XGBClassifier if TASK == "classification" else xgb.XGBRegressor
space = {"n_estimators": [200, 400, 700], "max_depth": [3, 4, 6, 8], "learning_rate": [0.02, 0.05, 0.1],
         "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.5, 0.75, 1.0], "min_child_weight": [1, 5, 10]}
search = RandomizedSearchCV(XGB(tree_method="hist", random_state=0), space, n_iter=4 if FAST else 20, cv=3, scoring=scoring, random_state=0)
search.fit(encode(Xtr, FEATURES), ytr)
enter("XGBoost (tuned)", "Classic ML", lambda: Tabular(XGB(tree_method="hist", random_state=0, **search.best_params_)).fit(Xtr, ytr), "random search")

In [ ]:
# @title The search, one line per configuration { display-mode: "form" }
trials = pd.DataFrame(search.cv_results_["params"]).assign(score=search.cv_results_["mean_test_score"])
fig = px.parallel_coordinates(trials, color="score", color_continuous_scale="Viridis",
                              title=f"XGBoost search · {len(trials)} trials · colour = cross-validated {scoring}")
fig.update_layout(height=420); fig.show()
tfm_ok = [r for r in RACE if r["status"] == "ok" and r["family"].startswith("Foundation")]
lora_ok = [r for r in tfm_ok if "peft" in TABTUNE_MODELS.get(r["model"], {}).get("strategies", "")]
LORA_BASE = max(lora_ok, key=lambda r: r[PRIMARY])["model"] if lora_ok else None
print("LoRA candidate:", LORA_BASE or f"none of the raced models has a LoRA path for {TASK} in this TabTune version")

In [ ]:
if RUN_PEFT and LORA_BASE:                                              # LoRA adapters on the best LoRA-capable model
    lora = TabularPipeline(model_name=LORA_BASE, task_type=TASK, tuning_strategy="peft",
                           tuning_params={"device": DEVICE, "epochs": LORA_EPOCHS, "learning_rate": 2e-5})
    enter(f"{LORA_BASE} + LoRA", "Foundation model (fine-tuned)", lambda: lora.fit(Xtr, ytr), f"LoRA, {LORA_EPOCHS} epoch(s)")

In [ ]:
from tabtune.ensemble import TabularEnsemble
members = [r["model"] for r in sorted(tfm_ok, key=lambda r: -r[PRIMARY])[:3]]
if RUN_ENSEMBLE and len(members) >= 2:                                  # greedy ensemble of the top foundation models
    ensemble = TabularEnsemble(models=[{"model_name": m, "tuning_strategy": "inference", "tuning_params": {"device": DEVICE}} for m in members],
                               ensemble_strategy="greedy_selection", task_type=TASK, verbose=False, **({"metric": "r2"} if TASK == "regression" else {}))
    enter("TabTune ensemble", "Ensemble", lambda: ensemble.fit(Xtr, ytr), "greedy selection of " + ", ".join(members))

## 4.4 Model selection
**Rule:** the highest validation score wins. Models within 0.002 of the best are treated as a tie, and the faster one wins, because the difference is noise and scoring cost is not. `REQUIRE_COMMERCIAL = True` restricts the choice to models whose weights are cleared for commercial use. The test set is not used.

In [ ]:
# @title Selection rule { display-mode: "form" }
TIE_MARGIN = 0.002

def select_champion(board, tie_margin=TIE_MARGIN, require_commercial=False):
    ok = board[board.status == "ok"].sort_values(PRIMARY, ascending=False)
    eligible = ok[ok.commercial_use != "no"] if require_commercial else ok
    best = eligible.iloc[0]
    tied = eligible[eligible[PRIMARY] >= best[PRIMARY] - tie_margin].sort_values("ms_per_row")
    winner = tied.iloc[0]["model"]
    why = (f"highest {PRIMARY} ({best[PRIMARY]:.4f})" if winner == best["model"] else
           f"tied with {best['model']} (within {tie_margin}) and {best['ms_per_row'] / tied.iloc[0]['ms_per_row']:.1f}× faster to score")
    return winner, why

In [ ]:
board = pd.DataFrame(RACE)
CHAMPION, why = select_champion(board, require_commercial=REQUIRE_COMMERCIAL)
champion = FITTED[CHAMPION]
print(f"selected {CHAMPION}: {why}")

In [ ]:
# @title Leaderboard dashboard { display-mode: "form" }
ok = board[board.status == "ok"].sort_values(PRIMARY, ascending=False).reset_index(drop=True)
FAM = {"Foundation model (zero-shot)": BLUE, "Foundation model (fine-tuned)": TEAL, "Classic ML": GREY, "Ensemble": ORANGE}
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Validation {PRIMARY}", "Accuracy vs scoring cost"))
o = ok.sort_values(PRIMARY)
fig.add_trace(go.Bar(x=o[PRIMARY], y=o.model, orientation="h", marker_color=[FAM[f] for f in o.family],
                     marker_line=dict(color=[DARK if m == CHAMPION else "rgba(0,0,0,0)" for m in o.model], width=3),
                     text=o[PRIMARY], texttemplate="%{text:.4f}", showlegend=False), 1, 1)
for fam, g in ok.groupby("family"):
    fig.add_trace(go.Scatter(x=g.ms_per_row, y=g[PRIMARY], mode="markers+text", text=g.model, textposition="top center", name=fam, marker=dict(size=14, color=FAM[fam])), 1, 2)
fig.update_xaxes(range=[ok[PRIMARY].min() - 0.03, ok[PRIMARY].max() + 0.02], row=1, col=1)
fig.update_xaxes(type="log", title_text="milliseconds per scored row", row=1, col=2)
fig.update_layout(template="plotly_white", height=520, title=f"AutoML leaderboard · selected: {CHAMPION} ({why})", legend=dict(orientation="h", y=-0.15))
fig.show()
display(board.style.format(precision=4, na_rep="-").hide(axis="index"))
audit("automl", "model selection", f"{len(ok)} models scored on validation; best {ok.iloc[0]['model']} at {ok.iloc[0][PRIMARY]:.4f}", f"SELECTED {CHAMPION}: {why}")
classic = ok[ok.family == "Classic ML"]
record("selection", candidates_raced=len(board), candidates_succeeded=len(ok), selected_model=CHAMPION, selection_metric=PRIMARY,
       selected_validation_score=float(ok.set_index("model").loc[CHAMPION, PRIMARY]), selection_reason=why,
       best_classic_model=str(classic.iloc[0]["model"]) if len(classic) else "none",
       best_classic_validation_score=float(classic[PRIMARY].max()) if len(classic) else float("nan"),
       selection_rule=f"highest validation score; ties within {TIE_MARGIN} go to the faster model", xgboost_trials=len(trials))
record("training", context_rows=len(Xtr), features=len(FEATURES), adaptation=str(ok.set_index("model").loc[CHAMPION, "tuning"]), device=GPU)

## 4.5 Global explainability
Permutation importance: shuffle one feature at a time on validation rows and measure how much the score falls. It is model-agnostic, so the same method explains a foundation model and a tree. (`pipeline.get_feature_importance()` is TabTune's built-in version.)

In [ ]:
# @title Permutation importance helper { display-mode: "form" }
def permutation_importance_of(model, X, y, feats, repeats=2, n=300):
    idx = X.sample(min(n, len(X)), random_state=0).index
    Xs, ys = X.loc[idx], y.loc[idx]
    base = metric(ys, score_of(model, Xs))
    out = {}
    for f in feats:
        drops = []
        for r in range(repeats):
            Xp = Xs.copy(); Xp[f] = Xs[f].sample(frac=1, random_state=r).to_numpy()
            drops.append(base - metric(ys, score_of(model, Xp)))
        out[f] = float(np.mean(drops))
    return pd.Series(out).sort_values(ascending=False)

EXPLAIN_FEATURES = [c for c in MI.index if c in FEATURES][:8 if FAST else 20]

In [ ]:
gimp = permutation_importance_of(champion, Xva, yva, EXPLAIN_FEATURES)
gimp.head(10)

In [ ]:
# @title What drives the selected model overall { display-mode: "form" }
t = gimp.head(15)[::-1]
fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], subplot_titles=(f"Drop in {PRIMARY} when a feature is shuffled", "Model importance vs raw signal (rank agreement)"))
fig.add_trace(go.Bar(x=t.values, y=[c[:30] for c in t.index], orientation="h", marker_color=BLUE, showlegend=False), 1, 1)
ranks = pd.DataFrame({"model": gimp.rank(ascending=False), "signal": MI.reindex(gimp.index).rank(ascending=False)})
fig.add_trace(go.Scatter(x=ranks.signal, y=ranks.model, mode="markers+text", text=[c[:14] for c in ranks.index], textposition="top center", marker=dict(size=9, color=ORANGE), showlegend=False), 1, 2)
fig.update_xaxes(title_text="rank by mutual information", row=1, col=2); fig.update_yaxes(title_text="rank by model importance", row=1, col=2)
fig.update_layout(template="plotly_white", height=560, title=f"Global explanation · {CHAMPION}")
fig.show()
record("explainability", method="permutation importance on validation rows; occlusion for single decisions", top_drivers=", ".join(gimp.head(5).index))

# 5 · Model testing
Everything in this section uses the **test set**, which no step above has touched.

In [ ]:
p_te = np.asarray(champion.predict(Xte))
y = yte.to_numpy()

In [ ]:
# @title Evaluation dashboard on the held-out test set { display-mode: "form" }
p_va = VAL_SCORES[CHAMPION]
TEST_SCORES = {m: score_of(FITTED[m], Xte) for m in ok.model.head(4) if m != CHAMPION}
TEST_SCORES[CHAMPION] = p_te
if TASK == "classification":
    grid = np.linspace(0.02, 0.98, 97)
    THRESHOLD = float(grid[np.argmax([skm.f1_score(yva, VAL_SCORES[CHAMPION] >= t) for t in grid])])          # chosen on validation
    pred = (p_te >= THRESHOLD).astype(int)
    perf = {"roc_auc": skm.roc_auc_score(y, p_te), "average_precision": skm.average_precision_score(y, p_te), "f1": skm.f1_score(y, pred),
            "precision": skm.precision_score(y, pred, zero_division=0), "recall": skm.recall_score(y, pred), "accuracy": skm.accuracy_score(y, pred),
            "balanced_accuracy": skm.balanced_accuracy_score(y, pred), "mcc": skm.matthews_corrcoef(y, pred), "brier": skm.brier_score_loss(y, p_te),
            "log_loss": skm.log_loss(y, np.clip(p_te, 1e-6, 1 - 1e-6)), "ks": float(np.max(np.abs(np.cumsum(y[np.argsort(-p_te)]) / y.sum() - np.cumsum(1 - y[np.argsort(-p_te)]) / (1 - y).sum()))),
            "gini": 2 * skm.roc_auc_score(y, p_te) - 1, "threshold": THRESHOLD}
    fig = make_subplots(rows=3, cols=3, vertical_spacing=0.1, horizontal_spacing=0.08,
                        subplot_titles=("ROC · all finalists", "Precision–recall · all finalists", "Calibration", "Score distribution by outcome",
                                        "Cumulative gains", "Lift by decile", f"Confusion matrix @ {THRESHOLD:.2f}", "KS: separation of the two classes", "Precision, recall, F1 by threshold"))
    for i, (m, s) in enumerate(TEST_SCORES.items()):
        fpr, tpr, _ = skm.roc_curve(y, s); pr, rc, _ = skm.precision_recall_curve(y, s)
        st = dict(color=BLUE if m == CHAMPION else PALETTE[(i + 1) % 8], width=3.5 if m == CHAMPION else 1.5)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{m} ({skm.roc_auc_score(y, s):.3f})", line=st), 1, 1)
        fig.add_trace(go.Scatter(x=rc, y=pr, line=st, showlegend=False), 1, 2)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(color=GREY, dash="dot"), showlegend=False), 1, 1)
    cal = pd.DataFrame({"p": p_te, "y": y, "b": pd.qcut(p_te, 10, labels=False, duplicates="drop")}).groupby("b").mean()
    fig.add_trace(go.Scatter(x=cal.p, y=cal.y, mode="lines+markers", line=dict(color=TEAL, width=3), showlegend=False), 1, 3)
    fig.add_trace(go.Scatter(x=[0, cal.p.max()], y=[0, cal.p.max()], line=dict(color=GREY, dash="dot"), showlegend=False), 1, 3)
    for cls, col in ((0, BLUE), (1, RED)):
        fig.add_trace(go.Histogram(x=p_te[y == cls], nbinsx=40, opacity=0.6, marker_color=col, histnorm="probability", showlegend=False), 2, 1)
    order = np.argsort(-p_te); share = np.arange(1, len(y) + 1) / len(y); gains = np.cumsum(y[order]) / y.sum()
    fig.add_trace(go.Scatter(x=share, y=gains, line=dict(color=BLUE, width=3), showlegend=False), 2, 2)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(color=GREY, dash="dot"), showlegend=False), 2, 2)
    dec = pd.DataFrame({"d": pd.qcut(pd.Series(p_te).rank(method="first", ascending=False), 10, labels=False), "y": y}).groupby("d").y.mean() / y.mean()
    fig.add_trace(go.Bar(x=[f"D{i + 1}" for i in dec.index], y=dec.values, marker_color=ORANGE, showlegend=False), 2, 3)
    cm = skm.confusion_matrix(y, pred)
    fig.add_trace(go.Heatmap(z=cm[::-1], x=[f"pred {c}" for c in CLASS_NAMES], y=[f"true {c}" for c in CLASS_NAMES[::-1]], text=cm[::-1], texttemplate="%{text}", colorscale="Blues", showscale=False), 3, 1)
    cum1, cum0 = np.cumsum(y[order]) / y.sum(), np.cumsum(1 - y[order]) / (1 - y).sum()
    fig.add_trace(go.Scatter(x=share, y=cum1, line=dict(color=RED), showlegend=False), 3, 2); fig.add_trace(go.Scatter(x=share, y=cum0, line=dict(color=BLUE), showlegend=False), 3, 2)
    fig.add_trace(go.Scatter(x=grid, y=[skm.precision_score(y, p_te >= t, zero_division=0) for t in grid], name="precision", line=dict(color=TEAL)), 3, 3)
    fig.add_trace(go.Scatter(x=grid, y=[skm.recall_score(y, p_te >= t) for t in grid], name="recall", line=dict(color=ORANGE)), 3, 3)
    fig.add_trace(go.Scatter(x=grid, y=[skm.f1_score(y, p_te >= t) for t in grid], name="F1", line=dict(color=DARK, width=3)), 3, 3)
    fig.add_vline(x=THRESHOLD, line_dash="dot", line_color=RED, row=3, col=3)
    fig.update_layout(barmode="overlay")
else:
    res = y - p_te
    perf = {"r2": skm.r2_score(y, p_te), "rmse": float(np.sqrt(np.mean(res ** 2))), "mae": float(np.mean(np.abs(res))), "median_abs_error": float(np.median(np.abs(res))),
            "mape_pct": float(np.mean(np.abs(res) / np.maximum(np.abs(y), 1e-9)) * 100), "explained_variance": skm.explained_variance_score(y, p_te),
            "max_error": float(np.max(np.abs(res))), "within_20pct_share": float(np.mean(np.abs(res) / np.maximum(np.abs(y), 1e-9) <= 0.2) * 100)}
    ape = np.abs(res) / np.maximum(np.abs(y), 1e-9) * 100
    fig = make_subplots(rows=2, cols=3, vertical_spacing=0.14, subplot_titles=("Predicted vs actual", "Residuals vs prediction", "Residual distribution",
                                                                               "Median % error by actual-value decile", "Share of predictions within an error band", "R² of the finalists"))
    lo, hi = np.percentile(np.r_[y, p_te], [1, 99])
    fig.add_trace(go.Scattergl(x=y, y=p_te, mode="markers", marker=dict(size=4, color=BLUE, opacity=0.4), showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], line=dict(color=DARK, dash="dot"), showlegend=False), 1, 1)
    fig.add_trace(go.Scattergl(x=p_te, y=res, mode="markers", marker=dict(size=4, color=ORANGE, opacity=0.4), showlegend=False), 1, 2)
    fig.add_trace(go.Histogram(x=res, nbinsx=60, marker_color=GREY, showlegend=False), 1, 3)
    bd = pd.DataFrame({"d": pd.qcut(y, 10, labels=False, duplicates="drop"), "ape": ape}).groupby("d").ape.median()
    fig.add_trace(go.Bar(x=[f"D{int(i) + 1}" for i in bd.index], y=bd.values, marker_color=TEAL, showlegend=False), 2, 1)
    bands = [10, 20, 35, 50]
    fig.add_trace(go.Bar(x=[f"within {b}%" for b in bands], y=[float(np.mean(ape <= b) * 100) for b in bands], marker_color=BLUE, showlegend=False,
                         text=[f"{np.mean(ape <= b) * 100:.0f}%" for b in bands], textposition="outside"), 2, 2)
    r2s = {m: skm.r2_score(y, s) for m, s in TEST_SCORES.items()}
    fig.add_trace(go.Bar(x=list(r2s), y=list(r2s.values()), marker_color=[BLUE if m == CHAMPION else GREY for m in r2s], showlegend=False,
                         text=[f"{v:.3f}" for v in r2s.values()], textposition="outside"), 2, 3)
fig.update_layout(template="plotly_white", height=980 if TASK == "classification" else 760, title=f"Test-set evaluation · {CHAMPION} · {len(y):,} unseen rows",
                  legend=dict(orientation="h", y=-0.06))
fig.show()
kp = list(perf.items())[:6]
fig = make_subplots(rows=1, cols=len(kp), specs=[[{"type": "indicator"}] * len(kp)])
for i, (k, v) in enumerate(kp, 1):
    fig.add_trace(go.Indicator(mode="number", value=v, title={"text": k.replace("_", " ")}, number={"valueformat": ",.3f" if abs(v) < 100 else ",.0f", "font": {"size": 40}}), 1, i)
fig.update_layout(height=200, margin=dict(t=50, b=10)); fig.show()
record("performance", evaluated_on="held-out test set", test_rows=len(y), **perf)

In [ ]:
# @title Interactive: move the decision threshold (classification) or the tolerance (regression) { display-mode: "form" }
def explore(value):
    if TASK == "classification":
        pr = (p_te >= value).astype(int); cm = skm.confusion_matrix(y, pr)
        txt = (f"threshold {value:.2f} · flagged {pr.mean():.1%} of applications · precision {skm.precision_score(y, pr, zero_division=0):.3f} · "
               f"recall {skm.recall_score(y, pr):.3f} · F1 {skm.f1_score(y, pr):.3f}")
        f = go.Figure(go.Heatmap(z=cm[::-1], x=[f"pred {c}" for c in CLASS_NAMES], y=[f"true {c}" for c in CLASS_NAMES[::-1]], text=cm[::-1],
                                 texttemplate="%{text}", colorscale="Blues", showscale=False))
    else:
        tol = value; a = np.abs(y - p_te) / np.maximum(np.abs(y), 1e-9) * 100
        txt = f"tolerance ±{tol:.0f}% · {np.mean(a <= tol):.1%} of recommendations are inside it"
        f = go.Figure(go.Histogram(x=np.clip(a, 0, 150), nbinsx=60, marker_color=[BLUE]))
        f.add_vline(x=tol, line_color=RED, line_dash="dot")
    f.update_layout(template="plotly_white", height=380, title=txt); f.show()

slider = W.FloatSlider(value=THRESHOLD, min=0.05, max=0.95, step=0.01, description="threshold") if TASK == "classification" \
    else W.FloatSlider(value=20, min=5, max=60, step=1, description="± % error")
W.interact(explore, value=slider);

In [ ]:
# @title Per-decision explanations (occlusion): replace one feature with a typical value and see how the prediction moves { display-mode: "form" }
TYPICAL = {f: (Xtr[f].median() if f not in CAT else Xtr[f].mode().iloc[0]) for f in FEATURES}
LOCAL_FEATURES = gimp.head(6 if FAST else 12).index.tolist()

def explain_case(row):
    variants = pd.concat([row.to_frame().T] * (len(LOCAL_FEATURES) + 1), ignore_index=True).astype(Xte.dtypes.to_dict())
    for i, f in enumerate(LOCAL_FEATURES, 1):
        variants.loc[i, f] = TYPICAL[f]
    s = score_of(champion, variants)
    return float(s[0]), pd.Series(s[0] - s[1:], index=LOCAL_FEATURES)

order_ = np.argsort(-p_te)
picks = {"highest prediction": order_[0], "second highest": order_[1], "median": order_[len(order_) // 2], "lowest prediction": order_[-1]}
fig, n_tr = go.Figure(), 0
for k, (label, i) in enumerate(picks.items()):
    base_, contrib = explain_case(Xte.iloc[i])
    c = contrib.reindex(contrib.abs().sort_values().index)
    fig.add_trace(go.Bar(x=c.values, y=[f"{f[:26]} = {Xte.iloc[i][f]:.3g}" if f not in CAT else f"{f[:22]} = {Xte.iloc[i][f]}" for f in c.index], orientation="h",
                         marker_color=[RED if v > 0 else BLUE for v in c.values], visible=k == 0,
                         name=f"{label}: prediction {base_:.3f}, actual {y[i]:.3g}"))
fig.update_layout(template="plotly_white", height=520, showlegend=False,
                  title="Why this prediction? Red pushed it up, blue pushed it down (vs a typical applicant)",
                  updatemenus=[dict(x=1, xanchor="right", y=1.18, buttons=[dict(label=l, method="update", args=[{"visible": [j == k for j in range(len(picks))]}])
                                                                               for k, l in enumerate(picks)])])
fig.show()

# 6 · Model inference in batch mode
The selected pipeline is saved as one artefact (preprocessing plus model), reloaded as a production job would, and used to score a file of new applications in chunks.

In [ ]:
ARTEFACT = "champion_pipeline.joblib"
if isinstance(champion, TabularPipeline):
    champion.save(ARTEFACT)
    scorer = TabularPipeline.load(ARTEFACT)
else:                                                       # a classic model or an ensemble won: plain joblib
    import joblib; joblib.dump(champion, ARTEFACT); scorer = joblib.load(ARTEFACT)
print(f"{ARTEFACT}: {os.path.getsize(ARTEFACT) / 1e6:.1f} MB")

In [ ]:
# @title Batch scoring helper and dashboard { display-mode: "form" }
def score_in_chunks(model, frame, n_chunks=5):
    timings, out = [], []
    for ch in np.array_split(np.arange(len(frame)), n_chunks):
        t0 = time.time(); out.append(score_of(model, frame.iloc[ch][FEATURES])); timings.append(len(ch) / (time.time() - t0))
    res = frame[[UID]].assign(prediction=np.concatenate(out))
    if TASK == "classification":
        res["decision"] = np.where(res.prediction >= THRESHOLD, "review / decline", "approve")
    return res, timings

In [ ]:
incoming = stream.head(N_MONTH)[[UID] + FEATURES]           # a file of new applications
scored, timings = score_in_chunks(scorer, incoming, n_chunks=5)
scored.to_csv("batch_scored.csv", index=False)
scored.head(8)

In [ ]:
# @title Throughput and output of the batch job { display-mode: "form" }
fig = make_subplots(rows=1, cols=2, subplot_titles=("Throughput per chunk (rows/second)", "Distribution of batch predictions"))
fig.add_trace(go.Bar(x=[f"chunk {i + 1}" for i in range(len(timings))], y=timings, marker_color=TEAL, showlegend=False), 1, 1)
fig.add_trace(go.Histogram(x=scored.prediction, nbinsx=40, marker_color=BLUE, showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=380, title=f"Batch scoring · {len(scored):,} rows → batch_scored.csv")
fig.show()
record("performance", batch_rows_per_second=float(np.median(timings)))
display(Markdown("**On the Lexsi platform** the same step is `project.upload_data('batch.csv', 'Batch1')` followed by "
                 "`project.model_inference(tag='Batch1', model_name=..., compute_type='T4.small')`, which returns the scored tag."))

# 7 · Model risk

## 7.1 Uncertainty
A single number hides how sure the model is. **Conformal prediction** turns each prediction into a set (classification) or an interval (regression) with a coverage promise that is then measured on the test set.

In [ ]:
from tabtune.uncertainty import ConformalClassifier, ConformalRegressor, Recalibrator
ALPHA = 0.10                                                     # promise: the actual amount is inside the interval 90% of the time
cp = ConformalRegressor(champion, method="absolute", alpha=ALPHA).calibrate(Xva, yva)
lo_, hi_ = cp.predict_interval(Xte)                             # per application: a low and a high amount
cov = cp.coverage(Xte, yte)
cov

In [ ]:
# @title Uncertainty dashboard { display-mode: "form" }
if TASK == "classification":
    sizes_ = np.asarray(cp.set_sizes(Xte))
    def ece(p, yy, bins=10):
        b = np.minimum((p * bins).astype(int), bins - 1)
        return float(sum(abs(p[b == k].mean() - yy[b == k].mean()) * (b == k).mean() for k in range(bins) if (b == k).any()))
    p_recal = p_te
    try:
        rc = Recalibrator(champion, method="temperature").fit(Xva, yva); p_recal = np.asarray(rc.predict_proba(Xte))[:, 1]
    except Exception as exc:
        print("recalibration skipped:", str(exc)[:100])
    unc = {"target_coverage": 1 - ALPHA, "measured_coverage": float(cov["coverage"]), "average_set_size": float(sizes_.mean()),
           "share_single_answer": float((sizes_ == 1).mean()), "share_needs_review": float((sizes_ != 1).mean()), "ece_before": ece(p_te, y), "ece_after_recalibration": ece(p_recal, y)}
    fig = make_subplots(rows=1, cols=3, subplot_titles=("Coverage: promised vs measured", "Prediction-set size", "Calibration error before and after"))
    fig.add_trace(go.Bar(x=["promised", "measured"], y=[1 - ALPHA, unc["measured_coverage"]], marker_color=[GREY, TEAL], text=[f"{1 - ALPHA:.0%}", f"{unc['measured_coverage']:.1%}"], textposition="outside", showlegend=False), 1, 1)
    vc = pd.Series(sizes_).value_counts().sort_index()
    fig.add_trace(go.Bar(x=[{0: "abstain", 1: "one answer", 2: "both possible"}.get(int(k), str(k)) for k in vc.index], y=vc.values, marker_color=[RED, BLUE, ORANGE][:len(vc)], showlegend=False), 1, 2)
    fig.add_trace(go.Bar(x=["raw", "temperature-scaled"], y=[unc["ece_before"], unc["ece_after_recalibration"]], marker_color=[GREY, TEAL], showlegend=False), 1, 3)
else:
    lo_, hi_ = np.asarray(lo_), np.asarray(hi_)
    unc = {"target_coverage": 1 - ALPHA, "measured_coverage": float(cov["coverage"]), "average_interval_width": float(np.mean(hi_ - lo_)),
           "interval_width_vs_median_target_pct": float(np.mean(hi_ - lo_) / np.median(y) * 100)}
    o = np.argsort(p_te)[:: max(1, len(y) // 150)]
    fig = make_subplots(rows=1, cols=2, column_widths=[0.3, 0.7], subplot_titles=("Coverage: promised vs measured", "Intervals for a sample of applicants (sorted by prediction)"))
    fig.add_trace(go.Bar(x=["promised", "measured"], y=[1 - ALPHA, unc["measured_coverage"]], marker_color=[GREY, TEAL], text=[f"{1 - ALPHA:.0%}", f"{unc['measured_coverage']:.1%}"], textposition="outside", showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=np.r_[np.arange(len(o)), np.arange(len(o))[::-1]], y=np.r_[hi_[o], lo_[o][::-1]], fill="toself", fillcolor="rgba(47,91,234,0.18)", line=dict(width=0), name="90% interval"), 1, 2)
    fig.add_trace(go.Scatter(x=np.arange(len(o)), y=p_te[o], line=dict(color=BLUE), name="prediction"), 1, 2)
    fig.add_trace(go.Scatter(x=np.arange(len(o)), y=y[o], mode="markers", marker=dict(color=ORANGE, size=5), name="actual"), 1, 2)
fig.update_layout(template="plotly_white", height=420, title="Uncertainty you can promise: split conformal prediction, checked on the test set")
fig.show()
record("uncertainty", method="split conformal prediction calibrated on the validation set", **unc)

## 7.2 Fairness

In [ ]:
# @title Group fairness on the test set { display-mode: "form" }
gcols = [c for c in test.columns if c.startswith(GROUP_PREFIX)]
if len(gcols) > 1:
    group = test[gcols].idxmax(axis=1).str.replace(GROUP_PREFIX, "")
elif gcols:
    group = test[gcols[0]].astype(str)
else:
    group = pd.Series("all", index=test.index)
group = group.where(group.map(group.value_counts()) >= 30, "other")
if TASK == "classification":
    g = pd.DataFrame({"g": group.values, "y": y, "flag": (p_te >= THRESHOLD).astype(int)})
    tab = g.groupby("g").apply(lambda d: pd.Series({"rows": len(d), "base_rate": d.y.mean(), "selection_rate": d.flag.mean(),
                                                    "true_positive_rate": d.flag[d.y == 1].mean(), "false_positive_rate": d.flag[d.y == 0].mean(),
                                                    "precision": d.y[d.flag == 1].mean() if d.flag.sum() else np.nan}))
    fair = {"demographic_parity_difference": float(tab.selection_rate.max() - tab.selection_rate.min()),
            "disparate_impact_ratio": float(tab.selection_rate.min() / max(tab.selection_rate.max(), 1e-9)),
            "equal_opportunity_difference": float(tab.true_positive_rate.max() - tab.true_positive_rate.min()),
            "false_positive_rate_difference": float(tab.false_positive_rate.max() - tab.false_positive_rate.min()),
            "equalized_odds_difference": float(max(tab.true_positive_rate.max() - tab.true_positive_rate.min(), tab.false_positive_rate.max() - tab.false_positive_rate.min())),
            "predictive_parity_difference": float(tab.precision.max() - tab.precision.min())}
    show = ["selection_rate", "true_positive_rate", "false_positive_rate", "precision"]
else:
    g = pd.DataFrame({"g": group.values, "y": y, "p": p_te})
    tab = g.groupby("g").apply(lambda d: pd.Series({"rows": len(d), "mean_actual": d.y.mean(), "mean_prediction": d.p.mean(),
                                                    "mae": (d.y - d.p).abs().mean(), "mean_error": (d.p - d.y).mean(), "mape_pct": ((d.y - d.p).abs() / d.y.abs().clip(lower=1e-9)).mean() * 100}))
    fair = {"mean_prediction_ratio": float(tab.mean_prediction.min() / tab.mean_prediction.max()), "mae_ratio": float(tab.mae.min() / tab.mae.max()),
            "mean_error_gap": float(tab.mean_error.max() - tab.mean_error.min()), "mape_gap_pct": float(tab.mape_pct.max() - tab.mape_pct.min()),
            "calibration_gap": float((tab.mean_prediction / tab.mean_actual).max() - (tab.mean_prediction / tab.mean_actual).min())}
    show = ["mean_actual", "mean_prediction", "mae", "mape_pct"]
fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.24, subplot_titles=(f"Rates by {GROUP_LABEL}", "Fairness metrics (0 = no gap; ratio: 1 = equal)"))
for i, m in enumerate(show):
    v = tab[m] / (tab[m].max() if TASK == "regression" else 1)
    fig.add_trace(go.Bar(x=tab.index, y=v, name=m.replace("_", " ") + (" (scaled)" if TASK == "regression" else ""), marker_color=PALETTE[i]), 1, 1)
fig.add_trace(go.Bar(x=list(fair.values()), y=[k.replace("_", " ") for k in fair], orientation="h", marker_color=GREY, showlegend=False,
                     text=[f"{v:.3f}" for v in fair.values()], textposition="outside"), 1, 2)
fig.update_xaxes(range=[0, max(fair.values()) * 1.25], row=1, col=2)
fig.update_layout(template="plotly_white", height=460, barmode="group", title=f"Fairness by {GROUP_LABEL} · {len(tab)} groups", legend=dict(orientation="h", y=-0.2))
fig.show()
display(tab.style.format(precision=3))
record("fairness", attribute=GROUP_LABEL, groups=len(tab), **fair)
record("limitations", note_1="fairness is measured on one attribute available in the data; other protected attributes were not available",
       note_2="zero-shot foundation models depend on the context rows; refresh the context when the population shifts",
       note_3="results come from a sample of the source data, not the full portfolio")

## 7.3 Regulatory evidence map, and accuracy checks
The table links each model-risk expectation to the evidence this notebook produced. The second cell runs sanity checks on the predictions themselves; the model card's own accuracy is checked in section 8.

In [ ]:
# @title Model risk: which evidence in this notebook supports which expectation { display-mode: "form" }
REG_MAP = pd.DataFrame([
    ("Conceptual soundness", "Problem statement, data preparation log, feature rationale, model selection rule", "SR 11-7 · SAMA model risk expectations"),
    ("Data quality and lineage", "Data profile, agent action log with human approvals, leakage checks", "SAMA Data Management · NDMO standards · PDPL"),
    ("Outcomes analysis", "Hold-out metrics, evaluation dashboards, segment performance", "SR 11-7 · Basel sound-practice principles"),
    ("Uncertainty", "Conformal prediction with measured coverage, calibration", "SR 11-7 (model limitations) · CBB AI policy"),
    ("Fairness", "Group metrics on protected or proxy attributes", "SDAIA AI Ethics Principles · CBB AI policy · EU AI Act"),
    ("Explainability", "Global drivers and per-decision explanations", "SDAIA AI Ethics Principles · EU AI Act"),
    ("Ongoing monitoring", "Drift monitoring, performance tracking, retraining with approval gate", "SR 11-7 · SAMA model risk expectations"),
    ("Documentation", "Generated model card with grounding check, audit trail", "SR 11-7 · ISO/IEC 42001"),
], columns=["Expectation", "Evidence produced in this notebook", "Frameworks it speaks to"])
display(REG_MAP.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
        .set_caption("Frameworks are named for orientation only. The bank's Compliance and Risk functions own the control mapping; "
                     "no clause-level compliance is claimed here."))

In [ ]:
# @title Prediction sanity checks (automatic accuracy guards) { display-mode: "form" }
checks = [("no missing predictions", bool(np.isfinite(p_te).all())),
          ("predictions in a valid range", bool(((p_te >= 0) & (p_te <= 1)).all()) if TASK == "classification" else bool((p_te > 0).mean() > 0.99)),
          ("deterministic: same rows give the same answer", bool(np.allclose(score_of(champion, Xte.head(40)), p_te[:40], rtol=1e-3, atol=1e-3))),
          ("beats the linear baseline on validation", bool(ok.set_index("model").loc[CHAMPION, PRIMARY] >= ok.set_index("model")[PRIMARY].get("Linear baseline", -np.inf))),
          ("test score close to validation score (no selection overfit)", bool(abs(metric(y, p_te) - ok.set_index("model").loc[CHAMPION, PRIMARY]) < 0.05)),
          ("conformal coverage within 3 points of the promise", bool(abs(unc["measured_coverage"] - (1 - ALPHA)) < 0.03))]
display(pd.DataFrame(checks, columns=["check", "passed"]).style.hide(axis="index").map(lambda v: f"color:{TEAL if v else RED};font-weight:600", subset=["passed"]))
record("governance", frameworks_referenced="SR 11-7; SAMA model risk expectations; SAMA and NDMO data standards; PDPL; CBB AI policy; SDAIA AI Ethics Principles; EU AI Act; ISO/IEC 42001",
       sanity_checks_passed=int(sum(v for _, v in checks)), sanity_checks_total=len(checks), human_approval_gates=sum(1 for a in AUDIT if a["actor"] == "human"))

# 8 · Artefacts before production

## 8.1 Model card
The model card is written by an LLM **only from the facts this notebook recorded**, against a documentation standard written for a model-risk committee. Four steps:

1. **Choose the writer.** The **Lexsi LLM gateway** (`/gateway/v1/chat/completions`): one API for every provider registered in the Lexsi project, with each call traced there. Section 1 sets it to Claude Sonnet 5.5 through Open Router, as registered in the Lexsi project; add the `LEXSI_API_TOKEN` Colab secret. Without them the notebook falls back to open weights on this GPU, then to a template. Only the aggregate fact sheet is sent, never customer rows.
2. **Send a full brief.** A system prompt (role, citation and number rules, what each of the 14 sections must contain, a self-check), the card template, the fact sheet grouped by section with a citation key per fact, and plain-English metric notes. Expand the prompt below to read exactly what is sent.
3. **Check automatically.** Every number must carry a citation key; each key must exist; each number must match its fact; all 14 sections must be present; no invented approvals, clause numbers, dates or compliance claims.
4. **Revise once.** If the checks fail, the issue list goes back to the writer for one rewrite. Whatever remains is listed for a human reviewer.

On the Lexsi platform this is the job of the Attest agent.

In [ ]:
# @title The documentation standard: system prompt, 14-section template, metric notes { display-mode: "form" }
CARD_SECTIONS = [
    "0. Document control", "1. Executive summary", "2. Intended use and scope", "3. Data", "4. Methodology and model selection",
    "5. Performance on held-out data", "6. Explainability", "7. Uncertainty and calibration", "8. Fairness",
    "9. Limitations and known risks", "10. Monitoring, alerting and retraining", "11. Regulatory alignment",
    "12. Approvals and sign-off", "13. Change log",
]

SYSTEM_PROMPT = textwrap.dedent("""
# ROLE
You are a senior model-risk documentation specialist at a Gulf bank supervised by SAMA (Saudi Central Bank) and the
Central Bank of Bahrain. You write the model card that goes to three readers:
- the model-risk committee, who decide whether the model may be used and read sections 0, 1, 9 and 12 first;
- the independent validator, who checks every number against the evidence;
- the supervisor's examiner, who looks for honest limitations and working controls.
A card that overstates, invents or cannot be traced is worse than no card. Precision and traceability come before style.

# SOURCE OF TRUTH
The user message contains a FACT SHEET. It is the only source you may use. Each fact has a key, for example
`performance.roc_auc = 0.7277`. The part before the dot is the fact group (data, preparation, selection, performance ...).

# CITATION RULES
1. Every number you write is followed immediately by the key it came from, in square brackets.
   Right:  The model reaches a ROC AUC of 0.728 [performance.roc_auc] on 3,000 unseen loans [performance.test_rows].
   Wrong:  The model reaches a ROC AUC of about 0.73 on the test set.            (no key; "about")
   Wrong:  ROC AUC improved by 0.015 over the baseline.                           (a number you computed)
2. Inside a table, put the key in the "Source" column of the same row instead of in brackets.
3. Use a key only for the fact it names. Never attach a key to a different number.
4. Text facts (model names, feature names, methods, rules) are quoted as written and also carry their key.

# NUMBER RULES
5. Round, never change: scores to 3 decimals, counts to whole numbers with thousands separators, percentages to 1 decimal.
6. A fact stored as a share (0.9017) may be written as a percentage (90.2%) with the same key. Say which it is.
7. Do NO arithmetic. Do not write differences, ratios, sums, averages or percentage changes that are not themselves facts.
   To compare two facts, state both with their keys and say which is higher, lower or about equal.
8. If something the template asks for is not in the fact sheet, write exactly "Not recorded". Never estimate or infer it.

# NEVER WRITE
9. Dates, people's names, team names, committee names, ticket numbers or version numbers that are not in the fact sheet.
10. Approvals, validation outcomes or sign-offs. Every approval is "Pending". The risk tier is "To be assigned by Model Risk Management".
11. Regulation article, clause or circular numbers, and any statement that the model "complies", "is compliant" or "meets"
    a regulation. Name a framework only if it appears in the fact sheet, and describe the evidence, not compliance.
12. Marketing language: "state-of-the-art", "robust", "best-in-class", "ensures", "guarantees", "cutting-edge".
13. Advice the facts do not support (for example recommending a threshold or a new feature).

# WHAT EACH SECTION MUST CONTAIN
0 Document control: the table, filled from use_case.* and the values marked "given".
1 Executive summary: exactly five bullets, in this order: what the model decides and for whom; which model was selected and
  how (selection.*); the headline held-out result with its metric explained in a few words (performance.*); how uncertainty
  is handled (uncertainty.*); the single most important risk to watch (limitations.* or monitoring.*). No bullet over 35 words.
2 Intended use and scope: the decision supported, intended users (only if recorded), the population the data covers, and an
  "Out of scope" list of at least three uses the facts do not support (other products, other populations, automated decisions
  without the recorded human review, and so on, phrased as limits of the evidence).
3 Data: source and size; a table of the data sets with rows and purpose; candidate features and what remained after
  preparation; what the data-preparation agent removed and why, including leakage controls (preparation.*); data-quality
  findings (missing values, skew); engineered features (features.*).
4 Methodology and model selection: how many candidates were raced and how many succeeded; the selection rule in plain words;
  the selected model and whether it is zero-shot, fine-tuned or trained; the hyperparameter search; why it won
  (selection.selection_reason if recorded); the hardware (training.*).
5 Performance on held-out data: the table with one row per performance.* fact and a plain-English "What it means" taken from
  the METRIC NOTES; then two or three sentences comparing the selected model with the best classic baseline (state both
  numbers, no arithmetic) and saying what the result means for the decision.
6 Explainability: the global method and the top drivers named in plain words; how single decisions are explained; a caution
  that importance shows association with the prediction, not cause.
7 Uncertainty and calibration: the method; promised versus measured coverage; what happens to uncertain cases; calibration
  figures if recorded.
8 Fairness: the table with one row per fairness.* fact; the attribute used and the number of groups; a neutral reading of the
  largest gap; a statement that other protected attributes were not assessed if the facts say so. No legal conclusions.
  For forecasting models write that fairness metrics do not apply and report consistency across segments instead.
9 Limitations and known risks: a numbered list of at least four items. Each item has three parts: the limitation, why it
  matters for the decision, and the control or monitor that addresses it (from monitoring.*, uncertainty.* or governance.*),
  or "No control recorded".
10 Monitoring, alerting and retraining: the monitors; the alert thresholds if recorded; how alerts reach the retraining
  agent; the champion-challenger rule and promotion margin; every human approval gate; then what monitoring has observed so
  far and every retraining event (retraining.*). If there are no observations yet, say "No production observations yet".
11 Regulatory alignment: the table. For each expectation list the card sections that hold the evidence, write the status as
  "Evidence provided" or "Not recorded", and cite one fact key. Close with one sentence naming the frameworks in
  governance.frameworks_referenced and stating that control mapping is owned by the bank's Compliance function.
12 Approvals and sign-off: the table with every cell "Pending", followed by "Open items for the validator": a bullet for
  every "Not recorded" you wrote in the card, or "None" if there were none.
13 Change log: every change supplied by the user, as bullets in the order given, or "Initial version".

# STYLE
- Plain English, short sentences, active voice. Explain each metric in a few words the first time it appears.
- Prefer tables for numbers and bullets for lists. No paragraph longer than four sentences.
- Feature names are written as recorded, in backticks.
- Target length: 900 to 1,500 words excluding tables.

# BEFORE YOU ANSWER, CHECK
- Every number has a key next to it or in its row, and the key exists in the fact sheet.
- No number was computed by you. No date, name, approval or clause number was invented.
- All 14 headings are present, numbered 0 to 13, in order, exactly as in the template.
- Every "Not recorded" in the card is listed under section 12.

# EXTRA RULES
- Square brackets contain ONLY a fact key, nothing else. Never write notes, reasoning or "not recorded" inside brackets.
  If the exact fact is missing, use the closest fact that is recorded with its own key, or write "Not recorded" in plain text.
- In section 11 the status may be "Evidence provided", "Partly provided: <which part>", "Not applicable: <why>" or "Not recorded".
  Use "Not recorded" only when no section of the card holds any evidence for that expectation.

# OUTPUT
Markdown only. Start with the line "# Model card · <use-case name>" and end with the last line of section 13.
No preamble, no closing remarks, no code fence around the card.
""").strip()

CARD_TEMPLATE = textwrap.dedent("""
# Model card · <use-case name>

## 0. Document control
| Field | Value | Source |
|---|---|---|
| Model name | <use-case name> | use_case.name |
| Model version | <version> | use_case.version |
| Card version | <card version given by the user> | given |
| Status | Draft, pending independent validation | given |
| Model owner | <owner> | use_case.owner |
| Risk tier | To be assigned by Model Risk Management | given |
| Written by | <writer given by the user> | given |

## 1. Executive summary
<five bullets as specified>

## 2. Intended use and scope
<decision supported, intended users, population in scope>

**Out of scope**
<at least three bullets>

## 3. Data
<source and size in one or two sentences>

| Data set | Rows | Used for | Source |
|---|---|---|---|
<one row per recorded data set>

<features before and after preparation; what the agent removed and why; leakage controls; data-quality findings; engineered features>

## 4. Methodology and model selection
<candidates, selection rule, selected model, adaptation, hyperparameter search, why it won, hardware>

## 5. Performance on held-out data
| Metric | Value | What it means | Source |
|---|---|---|---|
<one row per recorded performance fact>

<comparison with the best classic baseline, and what the result means for the decision>

## 6. Explainability
<global method and top drivers; single-decision explanations; the association-not-cause caution>

## 7. Uncertainty and calibration
<method, promised vs measured coverage, treatment of uncertain cases, calibration>

## 8. Fairness
| Metric | Value | Source |
|---|---|---|
<one row per recorded fairness fact, or a single row "Not recorded">

<neutral interpretation and limits>

## 9. Limitations and known risks
<numbered list; each item: limitation · why it matters · control or monitor>

## 10. Monitoring, alerting and retraining
<monitors, thresholds, alert routing, champion-challenger rule, approval gates>

**Observed so far**
<observations and retraining events, or "No production observations yet">

## 11. Regulatory alignment
| Expectation | Evidence in this card | Status | Source |
|---|---|---|---|
| Conceptual soundness | <sections> | <status> | <key> |
| Data quality and lineage | <sections> | <status> | <key> |
| Outcomes analysis | <sections> | <status> | <key> |
| Ongoing monitoring | <sections> | <status> | <key> |
| Explainability and fairness | <sections> | <status> | <key> |
| Documentation and change control | <sections> | <status> | <key> |

<one sentence naming the frameworks in the fact sheet; control mapping is owned by the bank's Compliance function>

## 12. Approvals and sign-off
| Role | Name | Decision | Date |
|---|---|---|---|
| Model owner | Pending | Pending | Pending |
| Independent model validation | Pending | Pending | Pending |
| Model risk committee | Pending | Pending | Pending |

**Open items for the validator**
<one bullet per "Not recorded" in this card, or "None">

## 13. Change log
<bullets>
""").strip()

# Plain-English meanings the writer may use in the "What it means" column (deliberately free of numbers)
METRIC_NOTES = {
    "roc_auc": "how well the model ranks risky cases above safe ones; higher is better",
    "average_precision": "how concentrated the true positives are among the highest scores; higher is better",
    "f1": "balance of precision and recall at the chosen threshold; higher is better",
    "precision": "of the cases the model flags, the share that are truly positive",
    "recall": "of the truly positive cases, the share the model flags",
    "accuracy": "share of all cases classified correctly at the chosen threshold",
    "balanced_accuracy": "accuracy averaged over the two classes, so the rarer class counts equally",
    "mcc": "single summary of the confusion matrix that stays fair when classes are unbalanced; higher is better",
    "brier": "average squared gap between predicted probability and outcome; lower is better",
    "log_loss": "penalty for confident wrong probabilities; lower is better",
    "ks": "largest separation between the score distributions of the two classes; higher is better",
    "gini": "a rescaling of ROC AUC used in credit scoring; higher is better",
    "threshold": "the score above which a case is flagged; chosen on validation data",
    "r2": "share of the variation in the target that the model explains; higher is better",
    "rmse": "typical size of the error, with large errors weighted more; in the target's units; lower is better",
    "mae": "average size of the error in the target's units; lower is better",
    "median_abs_error": "the error size that half of the predictions stay within; lower is better",
    "mape_pct": "average error as a percentage of the actual value; lower is better",
    "explained_variance": "share of the target's variation captured by the predictions; higher is better",
    "max_error": "the single largest error on the test set",
    "within_20pct_share": "percentage of predictions that land within one fifth of the actual value",
    "mase": "forecast error relative to a seasonal-naive forecast; below one beats it; lower is better",
    "wql": "error of the whole predicted range, not just the middle; lower is better",
    "smape": "symmetric percentage error of the median forecast; lower is better",
    "wape_pct": "total absolute error as a percentage of total actual flow; lower is better",
    "inside_90_band_pct": "percentage of actual outcomes that fell inside the ninety-percent band",
    "inside_80_band_pct": "percentage of actual outcomes that fell inside the eighty-percent band",
    "batch_rows_per_second": "scoring throughput in batch mode",
    "measured_coverage": "share of test cases whose true outcome fell inside the prediction set or interval",
    "disparate_impact_ratio": "lowest group selection rate divided by the highest; closer to one means more equal",
    "demographic_parity_difference": "gap between the highest and lowest group selection rates; closer to zero is more equal",
    "equal_opportunity_difference": "gap between groups in the share of true positives that are flagged",
    "equalized_odds_difference": "the larger of the gaps in true-positive and false-positive rates between groups",
    "predictive_parity_difference": "gap between groups in the precision of the flag",
    "max_feature_psi": "largest population stability index of any monitored feature; higher means more drift",
}

SECTION_GUIDE = {"use_case": "0, 1, 2", "data": "3", "preparation": "3", "features": "3", "selection": "4", "training": "4",
                 "performance": "1, 5", "explainability": "6", "uncertainty": "7", "fairness": "8", "limitations": "9",
                 "monitoring": "10", "retraining": "10, 13", "governance": "11"}


def flatten_facts(facts):
    flat = {}
    for sec, vals in facts.items():
        for k, v in vals.items():
            if isinstance(v, (np.floating, np.integer)):
                v = float(v)
            if isinstance(v, float) and not np.isfinite(v):
                continue
            flat[f"{sec}.{k}"] = v
    return flat


def build_messages(facts, changes, version_label, writer_label):
    flat = flatten_facts(facts)
    groups = {}
    for k, v in flat.items():
        groups.setdefault(k.split(".", 1)[0], []).append(f"- {k} = {v}")
    fact_block = "\n\n".join(f"[{g}]  (mainly for section {SECTION_GUIDE.get(g, 'as relevant')})\n" + "\n".join(lines) for g, lines in groups.items())
    notes = "\n".join(f"- {k}: {METRIC_NOTES[k.split('.', 1)[1]]}" for k in flat if k.split(".", 1)[1] in METRIC_NOTES)
    change_lines = "\n".join(f"- {c}" for c in changes) if changes else "- Initial version"
    user = (f"Write the model card, card version {version_label}. It is being written by: {writer_label}.\n\n"
            f"=== TEMPLATE (reproduce the headings and tables exactly; replace each <instruction>) ===\n\n{CARD_TEMPLATE}\n\n"
            f"=== FACT SHEET ({len(flat)} facts; the only source you may use; cite the key after every number) ===\n\n{fact_block}\n\n"
            f"=== METRIC NOTES (plain-English meanings for the 'What it means' column) ===\n{notes or '- none'}\n\n"
            f"=== CHANGES SINCE THE PREVIOUS VERSION (for section 13) ===\n{change_lines}\n")
    return SYSTEM_PROMPT, user


def writer_label():
    return f"{LEXSI_PROVIDER} · {LEXSI_MODEL} through the Lexsi LLM gateway" if WRITER == "lexsi" else (
        "open-weights model on the notebook GPU" if WRITER not in ("template",) else "template writer (no LLM)")


def show_prompt(version_label="v1", changes=None):
    system, user = build_messages(FACTS, changes, version_label, writer_label())
    display(Markdown(f"**Prompt sent to the writer:** system prompt {len(system):,} characters · user message {len(user):,} characters "
                     f"(about {(len(system) + len(user)) // 4:,} tokens) · {len(flatten_facts(FACTS))} facts · no customer-level data."))
    for title, body in (("System prompt: role, citation rules, section requirements, self-check", system),
                        ("User message: template, fact sheet, metric notes, changes", user)):
        display(HTML(f"<details><summary><b>{title}</b> · click to expand</summary>"
                     f"<pre style='white-space:pre-wrap;font-size:12px'>{_html.escape(body)}</pre></details>"))

In [ ]:
# @title Facts for the card, and free GPU memory for the writer { display-mode: "form" }
record("use_case", name="Home Credit · Loan Amount Recommendation", task=TASK, decision_supported="recommended credit amount with a range", owner="[model owner]", version="v1")
record("monitoring", plan="five monitors (data drift, prediction drift, performance, calibration, data quality) with warning and critical thresholds; "
       "critical alerts go to a retraining webhook; an agent retrains, verifies on unseen data and promotes only after human approval")
for _m in [m for m in list(FITTED) if m != CHAMPION]:           # release GPU memory held by the losing candidates
    FITTED.pop(_m, None)
import gc; gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

In [ ]:
show_prompt("v1")               # exactly what the writer receives

In [ ]:
# @title Writers (Lexsi gateway, local GPU, template) and the automatic checks { display-mode: "form" }
_LLM = {}
NEED_GB = {"gpt-oss-120b": 70, "32B": 70, "7B": 18}


def _load_local():
    import gc
    from transformers import AutoModelForCausalLM, AutoTokenizer
    for name in LOCAL_WRITERS if WRITER in ("local", "lexsi") else [WRITER]:
        need = next((g for k, g in NEED_GB.items() if k in name), 0)
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        if need and free_gb < need:
            print(f"skipping {name}: needs about {need} GB of free GPU memory, {free_gb:.0f} GB available")
            continue
        try:
            t0 = time.time(); gc.collect(); torch.cuda.empty_cache()
            tok = AutoTokenizer.from_pretrained(name)
            model = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto" if "gpt-oss" in name else torch.bfloat16, device_map="auto")
            _LLM.update(tok=tok, model=model, name=name, load_seconds=round(time.time() - t0, 1), gpu_gb=round(torch.cuda.memory_allocated() / 1e9, 1))
            print(f"loaded {name} in {_LLM['load_seconds']}s · GPU memory in use {_LLM['gpu_gb']} GB")
            return
        except Exception as exc:
            print(f"could not load {name} → trying the next option ({str(exc)[:120]})")
            _LLM.clear(); gc.collect(); torch.cuda.empty_cache()
    raise RuntimeError("no local writer could be loaded")


def _split_harmony(text):
    final = re.search(r"<\|channel\|>final<\|message\|>(.*?)(?:<\|return\|>|<\|end\|>|$)", text, flags=re.S)
    analysis = re.search(r"<\|channel\|>analysis<\|message\|>(.*?)(?:<\|end\|>|<\|start\|>|$)", text, flags=re.S)
    if final:
        return final.group(1).strip(), (analysis.group(1).strip() if analysis else "")
    return re.sub(r"<\|[^|]+\|>", "", text).strip(), ""


def write_local(system, messages):
    if "model" not in _LLM:
        _load_local()
    tok, model, name = _LLM["tok"], _LLM["model"], _LLM["name"]
    harmony = "gpt-oss" in name
    ids = tok.apply_chat_template([{"role": "system", "content": system}, *messages], add_generation_prompt=True,
                                  return_tensors="pt", **({"reasoning_effort": "medium"} if harmony else {})).to(model.device)
    out = model.generate(ids, max_new_tokens=9000 if harmony else 4000, do_sample=False)
    raw = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=not harmony)
    text, reasoning = _split_harmony(raw) if harmony else (raw.strip(), "")
    return text, reasoning, {"writer": name, "input_tokens": int(ids.shape[1]), "output_tokens": int(out.shape[1] - ids.shape[1]),
                             "load_seconds": _LLM["load_seconds"], "gpu_memory_gb": _LLM["gpu_gb"]}


def _as_prompt(system, messages):
    # text/generation takes one prompt string: the instructions, the task and, on the revision pass, the draft and the fixes
    parts = [f"### INSTRUCTIONS\n{system}"]
    for m in messages:
        title = "YOUR PREVIOUS DRAFT" if m["role"] == "assistant" else ("TASK" if len(parts) == 1 else "REVISION REQUEST")
        parts.append(f"### {title}\n{m['content']}")
    parts.append("### OUTPUT\nWrite the complete model card now, in Markdown, starting with the line '# Model card'.")
    return "\n\n".join(parts)


def _read(body):
    # text, token counts and extras from either response shape
    det = body.get("details")
    if isinstance(det, dict) and isinstance(det.get("result"), dict):                   # text/generation
        res = det["result"]
        return res.get("output") or "", (res.get("audit_trail") or {}).get("tokens") or {}, res
    if body.get("choices"):                                                               # chat/completions
        ch = body["choices"][0]; msg = ch.get("message") or {}
        content = msg.get("content") or ch.get("text") or ""
        if isinstance(content, list):
            content = "".join(b.get("text", "") for b in content if isinstance(b, dict))
        return content, body.get("usage") or {}, {"finish_reason": ch.get("finish_reason"),
                                                  "reasoning": msg.get("reasoning") or msg.get("reasoning_content") or ""}
    raise RuntimeError(f"no text in the gateway response: {str(det or body)[:300]}")    # e.g. blocked by a guardrail


def write_lexsi(system, messages):
    global LEXSI_ENDPOINT, LEXSI_REASONING
    t0, prompt = time.time(), _as_prompt(system, messages)
    chat_msgs = [{"role": "system", "content": system}, *messages]
    max_tokens = LEXSI_MAX_TOKENS
    for attempt in range(2):
        if LEXSI_ENDPOINT == "text":
            r = lexsi_generate(prompt, max_tokens)
            if r.status_code == 400 and "Provider must be one of" in r.text:
                print(f"text/generation does not accept provider '{LEXSI_PROVIDER}' on this gateway → same model through chat/completions")
                LEXSI_ENDPOINT = "chat"
        if LEXSI_ENDPOINT == "chat":
            r = lexsi_chat(chat_msgs, max_tokens, LEXSI_REASONING)
            if r.status_code in (400, 422) and LEXSI_REASONING:
                print(f"the call was rejected with the reasoning setting ({r.text[:120]}) → retrying with the model's default")
                LEXSI_REASONING = None
                r = lexsi_chat(chat_msgs, max_tokens)
        try:
            body = r.json()
        except ValueError:
            body = {"detail": r.text[:300]}
        if r.status_code != 200:
            hint = {401: "check the LEXSI_API_TOKEN secret", 403: "the token has no access to this project",
                    404: f"check that '{LEXSI_MODEL}' is registered in project {LEXSI_PROJECT} under provider '{LEXSI_PROVIDER}'",
                    504: "the gateway timed out; lower LEXSI_MAX_TOKENS or keep LEXSI_REASONING low"}.get(r.status_code, "see the gateway message")
            raise RuntimeError(f"Lexsi gateway returned {r.status_code}: {str(body.get('detail', body))[:300]} → {hint}")
        text, tokens, res = _read(body)
        finish = str(res.get("finish_reason") or "")
        if finish in ("length", "max_tokens") and attempt == 0:
            print(f"answer cut off at {max_tokens:,} tokens (thinking tokens count against the limit) → asking again with {max_tokens * 2:,}")
            max_tokens *= 2
            continue
        break
    if finish in ("length", "max_tokens"):
        print(f"⚠️ still cut off at {max_tokens:,} tokens; the checks below will list the missing sections")
    if prompt[-200:] in text:                                   # some text-generation backends echo the prompt
        text = text.split(prompt[-200:])[-1]
    if "# Model card" in text:
        text = text[text.index("# Model card"):]
    text = re.sub(r"^```(?:markdown)?\s*|\s*```$", "", text.strip())
    return text, res.get("reasoning") or "", {
        "writer": f"{LEXSI_PROVIDER} · {LEXSI_MODEL} (Lexsi, {'text/generation' if LEXSI_ENDPOINT == 'text' else 'chat/completions'})",
        "http_status": r.status_code, "http_version": r.http_version, "max_tokens_used": max_tokens, "reasoning_setting": str(LEXSI_REASONING),
        "input_tokens": tokens.get("prompt_tokens") or tokens.get("input_tokens") or 0,
        "output_tokens": tokens.get("completion_tokens") or tokens.get("output_tokens") or 0,
        "reasoning_tokens": tokens.get("reasoning_tokens") or (tokens.get("completion_tokens_details") or {}).get("reasoning_tokens"),
        "token_explanations_returned": any("expla" in k.lower() for k in res), "finish_reason": finish,
        "lexsi_project": LEXSI_PROJECT, "gateway_seconds": round(time.time() - t0, 1)}


def write_template(facts, changes, version_label):
    """Deterministic writer: fills the same template from the facts, with a citation after every number."""
    flat = flatten_facts(facts)
    def cite(key, label=None):
        v = flat.get(key)
        if v is None:
            return "Not recorded"
        if isinstance(v, float):
            v = f"{v:,.4g}" if abs(v) < 1000 else f"{v:,.0f}"
        return f"{v} [{key}]"
    rows = lambda sec: [(k.split(".", 1)[1], k) for k in flat if k.startswith(sec + ".")]
    table = lambda sec: "\n".join(f"| {name.replace('_', ' ')} | {cite(k)} | {k} |" for name, k in rows(sec)) or "| Not recorded | Not recorded | — |"
    uc = lambda k: flat.get(f"use_case.{k}", "Not recorded")
    L = [f"# Model card · {uc('name')}", "", "## 0. Document control", "| Field | Value | Source |", "|---|---|---|",
         f"| Model name | {uc('name')} | use_case.name |", f"| Model version | {uc('version')} | use_case.version |",
         f"| Card version | {version_label} | given |", "| Status | Draft, pending independent validation | given |",
         f"| Model owner | {uc('owner')} | use_case.owner |", "| Risk tier | To be assigned by Model Risk Management | given |",
         "| Written by | template writer (no LLM) | given |", "",
         "## 1. Executive summary", f"- Decision supported: {uc('decision_supported')}.",
         f"- Selected model: {cite('selection.selected_model')}.", f"- Selection metric: {cite('selection.selection_metric')}.",
         "- Held-out performance: see section 5.", "- Main risk to watch: drift in the population; see sections 9 and 10.", "",
         "## 2. Intended use and scope", f"- Task: {uc('task')}.", f"- Decision supported: {uc('decision_supported')}.",
         "- Out of scope: any use not described in the facts.", "",
         "## 3. Data", "| Item | Value | Source |", "|---|---|---|", table("data"), table("preparation"), table("features"), "",
         "## 4. Methodology and model selection", "| Item | Value | Source |", "|---|---|---|", table("selection"), table("training"), "",
         "## 5. Performance on held-out data", "| Metric | Value | What it means | Source |", "|---|---|---|---|",
         "\n".join(f"| {n.replace('_', ' ')} | {cite(k)} | — | {k} |" for n, k in rows("performance")) or "| Not recorded | — | — | — |", "",
         "## 6. Explainability", "| Item | Value | Source |", "|---|---|---|", table("explainability"), "",
         "## 7. Uncertainty and calibration", "| Item | Value | Source |", "|---|---|---|", table("uncertainty"), "",
         "## 8. Fairness", "| Metric | Value | Source |", "|---|---|---|", table("fairness"), "",
         "## 9. Limitations and known risks", *[f"{i}. {flat[k]}" for i, k in enumerate([k for k in flat if k.startswith("limitations.")], 1)], "",
         "## 10. Monitoring, alerting and retraining", "| Item | Value | Source |", "|---|---|---|", table("monitoring"), table("retraining"), "",
         "## 11. Regulatory alignment", "| Expectation | Evidence in this card | Status | Source |", "|---|---|---|---|",
         "| Conceptual soundness | Sections 3–4 | Evidence provided | selection.selection_rule |",
         "| Data quality and lineage | Section 3 | Evidence provided | preparation.features_removed |",
         "| Outcomes analysis | Section 5 | Evidence provided | performance.evaluated_on |",
         "| Ongoing monitoring | Section 10 | Evidence provided | monitoring.plan |",
         "| Explainability and fairness | Sections 6, 8 | Evidence provided | explainability.method |",
         "| Documentation and change control | Sections 12–13 | Evidence provided | governance.human_approval_gates |", "",
         f"Frameworks referenced: {flat.get('governance.frameworks_referenced', 'Not recorded')}. Control mapping is owned by the bank's Compliance function.", "",
         "## 12. Approvals and sign-off", "| Role | Name | Decision | Date |", "|---|---|---|---|",
         "| Model owner | Pending | Pending | Pending |", "| Independent model validation | Pending | Pending | Pending |",
         "| Model risk committee | Pending | Pending | Pending |", "",
         "## 13. Change log", *([f"- {c}" for c in changes] if changes else ["- Initial version"])]
    return "\n".join(L), "", {"writer": "template writer (no LLM)"}



FORBIDDEN = [(r"\b(article|clause|circular)\s+(no\.?\s*)?\d", "cites a regulation article or clause number"),
             (r"\b(approved|validated|signed off)\s+(by|on)\b", "claims an approval or validation"),
             (r"\b(fully|is)\s+compliant\b|\bcomplies with\b", "claims compliance"),
             (r"\b(19|20)\d{2}-\d{2}-\d{2}\b", "contains a date that is not in the facts")]


def _matches(value, written):
    """Does the number as written match the fact value (allowing rounding and share↔percent)?"""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return False
    d = len(written.split(".")[1]) if "." in written else 0
    n = float(written)
    tol = 0.5 * 10 ** (-d) + 1e-9
    return abs(v - n) <= tol or abs(v * 100 - n) <= tol or (abs(v) >= 1000 and abs(v - n) <= max(tol, abs(v) * 0.0005))


def check_card(card, facts, changes=None):
    """Automatic hallucination and completeness checks on a generated card."""
    flat = flatten_facts(facts)
    card = re.sub(r"\b[Ss]ections?\s+\d+(?:\s*[–-]\s*\d+)?(?:\s*(?:,|and)\s*\d+)*", "Section", card)   # section references are not facts
    text = card.replace(",", "")
    issues = []
    # 1. every cited key exists, and the number next to it matches the fact
    cites = list(re.finditer(r"\[([a-z_]+\.[a-z0-9_]+)\]", card))
    bad_keys = sorted({m.group(1) for m in cites if m.group(1) not in flat})
    mism = []
    for m in cites:
        k = m.group(1)
        if k not in flat:
            continue
        before = card[max(0, m.start() - 40):m.start()].replace(",", "")
        nums = re.findall(r"-?\d+(?:\.\d+)?", before)
        if nums and isinstance(flat[k], (int, float)) and not isinstance(flat[k], bool) and not _matches(flat[k], nums[-1]):
            mism.append(f"{nums[-1]} [{k}] (fact = {flat[k]})")
    # table rows: value cell vs source cell
    for row in re.findall(r"^\|(.+)\|\s*$", card, flags=re.M):
        cells = [c.strip() for c in row.split("|")]
        if len(cells) >= 3 and cells[-1] in flat and isinstance(flat[cells[-1]], (int, float)) and not isinstance(flat[cells[-1]], bool):
            nums = re.findall(r"-?\d+(?:\.\d+)?", " ".join(cells[1:-1]).replace(",", ""))
            if nums and not any(_matches(flat[cells[-1]], n) for n in nums):
                mism.append(f"table row '{cells[0]}': {nums[0]} vs {cells[-1]} = {flat[cells[-1]]}")
    # 2. numbers that are not grounded in any fact (prose only; ignore headings, list numbering and small integers)
    known = [v for v in flat.values() if isinstance(v, (int, float)) and not isinstance(v, bool)]
    known += [float(n) for v in flat.values() if isinstance(v, str) for n in re.findall(r"\d+(?:\.\d+)?", v.replace(",", ""))]
    known += [float(n) for k in flat for n in re.findall(r"\d+", k)]          # numbers that are part of a fact's name, e.g. the 80% band
    body = re.sub(r"^#+ .*$", "", text, flags=re.M)
    body = re.sub(r"^\s*\d+\.\s", "", body, flags=re.M)
    nums = [n for n in re.findall(r"(?<![\w.\[])-?\d+(?:\.\d+)?(?![\w\]])", body) if not (n.isdigit() and int(n) <= 13)]
    change_nums = set(re.findall(r"\d+(?:\.\d+)?", " ".join(changes or []).replace(",", "")))
    ungrounded = sorted({n for n in nums if n not in change_nums and not any(_matches(v, n) for v in known)})
    # 3. structure and forbidden content
    missing_sections = [s for s in CARD_SECTIONS if not re.search(r"^##\s*" + re.escape(s), card, flags=re.M)]
    fact_text = " ".join(str(v) for v in flat.values()).lower()
    forbidden = [why for pat, why in FORBIDDEN
                 if any(m.group(0).lower() not in fact_text for m in re.finditer(pat, card, flags=re.I))]     # allowed if it is a recorded fact
    notes = [m.group(0) for m in re.finditer(r"\[([^\]\n]{1,300})\]", card)
             if not re.fullmatch(r"[a-z_]+\.[a-z0-9_]+", m.group(1)) and "." in m.group(1) and "_" in m.group(1)]
    if notes: issues.append(f"brackets must contain only a fact key; rewrite these as plain text with a proper key: {'; '.join(n[:100] for n in notes[:5])}")
    if bad_keys: issues.append(f"cites keys that do not exist: {', '.join(bad_keys[:8])}")
    if mism: issues.append(f"numbers that do not match their cited fact: {'; '.join(mism[:8])}")
    if ungrounded: issues.append(f"numbers not found in the facts: {', '.join(ungrounded[:12])}")
    if missing_sections: issues.append(f"missing sections: {', '.join(missing_sections)}")
    issues += [f"forbidden content: {f}" for f in forbidden]
    report = {"citations": len(cites), "invalid_citation_keys": len(bad_keys), "malformed_citations": len(notes), "number_mismatches": len(mism),
              "ungrounded_numbers": len(ungrounded), "sections_present": f"{len(CARD_SECTIONS) - len(missing_sections)}/{len(CARD_SECTIONS)}",
              "forbidden_content": len(forbidden)}
    checked = max(len(nums) + len(cites), 1)
    report["grounded_pct"] = round(100 * (1 - (len(ungrounded) + len(mism)) / checked), 1)
    report["passed"] = not issues
    return report, issues


def generate_model_card(facts, changes=None, version_label="v1"):
    """Draft → automatic checks → one revision with the issue list (if needed) → final checks."""
    facts = json.loads(json.dumps(facts, default=float))
    trace, rounds = {"card_version": version_label}, []
    system, user = build_messages(facts, changes, version_label, writer_label())
    messages = [{"role": "user", "content": user}]
    write = write_lexsi if WRITER == "lexsi" else write_local
    t0 = time.time()
    try:
        if WRITER == "template":
            card, reasoning, meta = write_template(facts, changes, version_label)
        else:
            card, reasoning, meta = write(system, messages)
    except Exception as exc:
        print(f"⚠️ writer failed: {str(exc)[:500]}\n   → using the template writer for this card")
        card, reasoning, meta = write_template(facts, changes, version_label)
    report, issues = check_card(card, facts, changes)
    rounds.append({"round": "draft", **report})
    if issues and REVISION_PASS and not meta["writer"].startswith("template"):
        revise = ("An automatic checker compared your draft with the fact sheet and found these problems:\n- " + "\n- ".join(issues) +
                  "\n\nRewrite the COMPLETE card. Fix every problem listed: replace any number that is not in the fact sheet with the correct "
                  "fact and its key, or with 'Not recorded'; remove anything you computed yourself; restore any missing heading exactly as in "
                  "the template. Keep everything that was already correct. Same template, same rules, Markdown only.")
        messages += [{"role": "assistant", "content": card}, {"role": "user", "content": revise}]
        try:
            card2, reasoning2, meta2 = write(system, messages)
            report2, issues2 = check_card(card2, facts, changes)
            rounds.append({"round": "revision", **report2})
            if len(issues2) <= len(issues):
                card, issues, report = card2, issues2, report2
                reasoning = (reasoning + "\n\n--- revision ---\n" + reasoning2).strip()
                meta["output_tokens"] = meta.get("output_tokens", 0) + meta2.get("output_tokens", 0)
        except Exception as exc:
            print("revision pass failed:", str(exc)[:160])
    trace.update(meta, seconds=round(time.time() - t0, 1), rounds=len(rounds), prompt_characters=len(system) + len(user))
    trace["_reasoning"], trace["_rounds"], trace["_issues"] = reasoning, rounds, issues
    return card, trace, report


def show_card(card, trace, report):
    display(pd.DataFrame([{k: v for k, v in trace.items() if not k.startswith("_")}]).T.rename(columns={0: "trace"}))
    display(pd.DataFrame(trace["_rounds"]).set_index("round").T.rename_axis("automatic checks"))
    fig = go.Figure(go.Indicator(mode="gauge+number", value=report["grounded_pct"], number={"suffix": "%"},
                                 title={"text": "Numbers in the card that trace back to a recorded fact"},
                                 gauge={"axis": {"range": [0, 100]}, "bar": {"color": TEAL if report["passed"] else ORANGE}}))
    fig.update_layout(height=250, margin=dict(t=60, b=10)); fig.show()
    if trace["_issues"]:
        print("remaining issues for a human reviewer:\n- " + "\n- ".join(trace["_issues"]))
    if trace["_reasoning"]:
        display(HTML(f"<details><summary><b>Writer's reasoning trace</b> ({len(trace['_reasoning']):,} characters) · click to expand</summary>"
                     f"<pre style='white-space:pre-wrap;font-size:12px'>{_html.escape(trace['_reasoning'][:12000])}</pre></details>"))
    display(Markdown(card))

In [ ]:
CARD_V1, TRACE_V1, CHECKS_V1 = generate_model_card(FACTS, version_label="v1")

In [ ]:
show_card(CARD_V1, TRACE_V1, CHECKS_V1)
open("model_card_v1.md", "w").write(CARD_V1);

## 8.2 Production API contract
The request follows the Lexsi case-registration call (`register_case` in the Lexsi SDK: project, unique identifier, tag, data, compute and explanation methods). The cell validates a real request against the schema and answers it with the selected model, so the contract is tested, not just written.

In [ ]:
# @title The scoring endpoint behind the contract { display-mode: "form" }
SCHEMA = {f: ("string" if f in CAT else "number") for f in FEATURES}
def score_api(request):
    t0 = time.time()
    missing_ = [f for f in FEATURES if f not in request.get("data", [{}])[0]]
    if missing_:
        return {"success": False, "details": f"missing fields: {missing_[:5]}"}
    row = pd.DataFrame(request["data"])[FEATURES].astype(Xte.dtypes.to_dict())
    s = float(score_of(champion, row)[0]); _, contrib = explain_case(row.iloc[0])
    body = {"success": True, "model_name": CHAMPION, "model_version": "v1", "unique_identifier": request["unique_identifier"], "pred_value": round(s, 4)}
    if TASK == "classification":
        body.update(pred_category=CLASS_NAMES[int(s >= THRESHOLD)], decision="review / decline" if s >= THRESHOLD else "approve", threshold=round(THRESHOLD, 2))
    else:
        lo1, hi1 = cp.predict_interval(row); body.update(interval_90=[round(float(np.asarray(lo1)[0]), 2), round(float(np.asarray(hi1)[0]), 2)])
    body.update(top_drivers={k: round(float(v), 4) for k, v in contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(5).items()},
                audit_trail={"latency_ms": round((time.time() - t0) * 1000), "explanation_method": "occlusion", "scored_at": pd.Timestamp.now().isoformat(timespec="seconds")})
    return body
display(pd.DataFrame({"field": list(SCHEMA)[:12], "type": list(SCHEMA.values())[:12]}).T.style.set_caption(f"Input schema · {len(SCHEMA)} fields (first 12 shown)"))

In [ ]:
i = int(order_[0])
request = {"project_name": "[lexsi project]", "unique_identifier": str(test.iloc[i][UID]), "tag": "Live", "serverless_type": "nova-1",
           "xai": ["shap"], "data": [json.loads(Xte.iloc[[i]].to_json(orient="records"))[0]]}
response = score_api(request)
print("POST /case\n" + json.dumps({**request, "data": [f"… {len(FEATURES)} fields …"]}, indent=1))
print("\nRESPONSE\n" + json.dumps(response, indent=1))

In [ ]:
score_api({"unique_identifier": "x", "data": [{"wrong": 1}]})       # a malformed request is rejected

# 9 · Model in production

## 9.1 Monitoring options
A model in production can fail in different ways, and each needs its own monitor. This notebook runs all of the rows below every month and raises alerts from the rules in 9.2.

| Monitor | Question it answers | Measure used here | Needs outcomes? |
|---|---|---|---|
| **Data drift** | Do new applicants look like the ones the model learned from? | Population stability index (PSI) and Kolmogorov–Smirnov test per feature | No |
| **Prediction drift** | Is the model's output distribution moving? | PSI of this month's scores against the release scores; mean-score shift | No |
| **Performance** | Is the model still accurate? | R² on each month once outcomes are known, against the release level | Yes |
| **Calibration** | Do predicted values still mean what they say? | Mean bias as a % of the average amount | Yes |
| **Data quality** | Is the feed itself broken? | Rise in missing values; share of values outside the training range | No |
| **Fairness drift** | Are group gaps widening? | Gap in mean absolute error between groups, against the release gap | Yes |
| Concept drift | Has the link between inputs and outcome changed? | Shows up as performance decay with little data drift | Yes |

Alerts come in two levels. A **warning** notifies the model owner. A **critical** alert is sent to the **retraining webhook**, an HTTP endpoint that queues a job for the retraining agent. Data-quality alerts are routed to the data owner instead, because retraining on a broken feed would make things worse.

## 9.2 Production stream, alert rules and the monitor
The production stream is replayed as monthly batches. So the monitors have something to find, the applicant mix shifts a little more each month on the strongest feature (real rows, resampled; no values altered), and in month 4 a feed problem blanks part of one field.

In [ ]:
# @title Production stream: monthly batches with gradual drift and one feed problem { display-mode: "form" }
def psi(ref, new, bins=10):
    ref, new = pd.Series(ref).dropna(), pd.Series(new).dropna()
    if not len(ref) or not len(new):
        return 0.0
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    a = np.histogram(ref, edges)[0] / len(ref) + 1e-4; b = np.histogram(new, edges)[0] / len(new) + 1e-4
    return float(np.sum((b - a) * np.log(b / a)))

drift_feature = next(c for c in gimp.index if c in cont)
z = ((stream[drift_feature] - stream[drift_feature].mean()) / (stream[drift_feature].std() + 1e-9)).fillna(0).clip(-3, 3)
pool, months = stream.copy(), []
for m in range(MONTHS):
    pick = pool.sample(min(N_MONTH, len(pool)), weights=np.exp(0.35 * m * z.loc[pool.index]), random_state=m)
    pool = pool.drop(pick.index); months.append(pick.reset_index(drop=True))
MON_FEATURES = [c for c in gimp.index if c in NUM][:12]
QUALITY_FIELD = next((c for c in MON_FEATURES if c != drift_feature and c in cont), MON_FEATURES[-1])
if MONTHS >= 4:
    months[3].loc[months[3].sample(frac=0.12, random_state=1).index, QUALITY_FIELD] = np.nan
release_perf = metric(y, p_te)
print(f"{MONTHS} monthly batches of {N_MONTH:,} · gradual drift on '{drift_feature}' · feed problem on '{QUALITY_FIELD}' in month 4")

In [ ]:
ALERT_RULES = [   # warning → notify the owner · critical → the action
    dict(rule="feature drift",       monitor="data drift",       metric="max_feature_psi",        warn=0.10, critical=0.25, action="retrain"),
    dict(rule="many features drift", monitor="data drift",       metric="share_features_drifted", warn=0.15, critical=0.30, action="retrain"),
    dict(rule="prediction drift",    monitor="prediction drift", metric="prediction_psi",         warn=0.10, critical=0.25, action="retrain"),
    dict(rule="performance decay",   monitor="performance",      metric="performance_drop",       warn=0.02, critical=0.05, action="retrain"),
    dict(rule="calibration drift",   monitor="calibration",      metric="calibration_error",      warn=5.0, critical=10.0, action="retrain"),
    dict(rule="missing values",      monitor="data quality",     metric="missing_increase_pp",    warn=2.0,  critical=5.0,  action="block_and_notify"),
    dict(rule="out-of-range values", monitor="data quality",     metric="out_of_range_pct",       warn=2.0,  critical=5.0,  action="block_and_notify"),
    dict(rule="fairness gap",        monitor="fairness",         metric="fairness_gap_increase",  warn=0.05, critical=0.10, action="notify"),
]
pd.DataFrame(ALERT_RULES)

In [ ]:
# @title The monitor: every measure for a batch, then the alert rules { display-mode: "form" }
import hmac, hashlib, queue, threading, uuid, secrets as _secrets
import requests
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer


from scipy.stats import ks_2samp


def evaluate_rules(metrics, rules, period, model_version):
    alerts = []
    for r in rules:
        v = metrics.get(r["metric"])
        if v is None or not np.isfinite(v):
            continue
        sev = "critical" if v >= r["critical"] else "warning" if v >= r["warn"] else None
        if sev:
            alerts.append({"alert_id": f"{period}:{r['rule']}", "rule": r["rule"], "monitor": r["monitor"], "severity": sev,
                           "action": r["action"] if sev == "critical" else "notify", "period": period, "metric": r["metric"],
                           "value": round(float(v), 4), "threshold": r["critical"] if sev == "critical" else r["warn"],
                           "model_version": model_version, "raised_at": pd.Timestamp.now().isoformat(timespec="seconds")})
    return alerts


def _ece(p, yy, bins=10):
    p, yy = np.asarray(p), np.asarray(yy); b = np.minimum((p * bins).astype(int), bins - 1)
    return float(sum(abs(p[b == k].mean() - yy[b == k].mean()) * (b == k).mean() for k in range(bins) if (b == k).any()))


def _group_gap(frame, scores):
    gcols = [c for c in frame.columns if c.startswith(GROUP_PREFIX)]
    if not gcols:
        return np.nan
    g = (frame[gcols].idxmax(axis=1) if len(gcols) > 1 else frame[gcols[0]].astype(str)).to_numpy()
    if TASK == "classification":
        rate = pd.Series(np.asarray(scores) >= THRESHOLD).groupby(g).mean()
        return float(rate.max() - rate.min())
    err = pd.Series(np.abs(frame[TARGET_COL].to_numpy() - np.asarray(scores))).groupby(g).mean()
    return float(err.max() / max(err.min(), 1e-9) - 1)


class ModelMonitor:
    def __init__(self, model, reference, rules, version=1):
        self.rules, self.history, self.alerts = rules, [], []
        self.promote(model, reference, version)

    def promote(self, model, reference, version):
        """Point the monitor at a new live model and the data it was built on."""
        self.model, self.reference, self.version = model, reference, version
        self.ref_scores = score_of(model, Xva)
        self.ref_missing = reference[MON_FEATURES].isna().mean() * 100
        self.ref_lo, self.ref_hi = reference[MON_FEATURES].min(), reference[MON_FEATURES].max()
        self.ref_gap = _group_gap(test, score_of(model, Xte))

    def check(self, batch, period):
        s = score_of(self.model, batch[FEATURES]); yb = batch[TARGET_COL].to_numpy()
        f_psi = {f: psi(self.reference[f], batch[f]) for f in MON_FEATURES}
        ks_p = {f: ks_2samp(self.reference[f].dropna(), batch[f].dropna()).pvalue for f in MON_FEATURES if batch[f].notna().sum() > 5}
        perf = metric(yb, s)
        m = {"period": period, "model_version": self.version,
             "max_feature_psi": max(f_psi.values()), "share_features_drifted": float(np.mean([v > 0.2 for v in f_psi.values()])),
             "features_ks_p_below_0_01": int(sum(p < 0.01 for p in ks_p.values())),
             "prediction_psi": psi(self.ref_scores, s), "mean_score": float(np.mean(s)),
             "performance": perf, "performance_drop": release_perf - perf,
             "calibration_error": _ece(s, yb) if TASK == "classification" else float(abs(np.mean(s - yb)) / np.mean(np.abs(yb)) * 100),
             "missing_increase_pp": float((batch[MON_FEATURES].isna().mean() * 100 - self.ref_missing).max()),
             "out_of_range_pct": float(((batch[MON_FEATURES] < self.ref_lo) | (batch[MON_FEATURES] > self.ref_hi)).mean().max() * 100),
             "fairness_gap_increase": float(_group_gap(batch, s) - self.ref_gap)}
        alerts = evaluate_rules(m, self.rules, period, f"v{self.version}")
        self.history.append({**m, "feature_psi": f_psi, "alerts": len(alerts)})
        self.alerts += alerts
        return {"metrics": m, "alerts": alerts}

In [ ]:
monitor = ModelMonitor(champion, reference=ctx, rules=ALERT_RULES)

## 9.3 The retraining webhook and agent
The retraining agent does not poll anything: it has an **inbox**, a small HTTP endpoint (a webhook). The monitoring system POSTs every alert to it, signed with a shared secret. The webhook:

- rejects anything without a valid signature (401) or with missing fields (422), and ignores duplicates;
- logs warnings and non-retraining actions and notifies the owner (200);
- for a **critical** alert whose action is `retrain`, opens one retraining job per period and queues it (202), however many alerts point at the same month.

The agent then takes each queued job: **observe** (read the alerts) → **decide** → **human approval** → **retrain** a challenger with the newest labelled data → **verify** champion vs challenger on the next, unseen month → **human approval** → **promote**, and re-point the monitor at the new model. In production the webhook would run as a service (for example behind the bank's API gateway); here it runs inside the notebook on a local port.

**With `AGENT_MODE = "llm"`** (section 1) the agent is an LLM. Its tools: `read_job`, `retrain_challenger`, `verify_on_next_month` and `promote_challenger`. It decides whether the alerts justify retraining (data-quality alerts do not), which challengers to build, and whether to promote. The promotion rule is enforced inside the promote tool, so the agent cannot promote a challenger that has not won on unseen data. Retraining and promotion wait for human approval.

In [ ]:
# @title Webhook: the retraining agent's inbox (HTTP server, signature check, routing, job queue) { display-mode: "form" }
import hmac, hashlib, queue, threading, uuid, secrets as _secrets
import requests
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

WEBHOOK_SECRET = _secrets.token_hex(16)      # shared between the monitoring system and the webhook


class RetrainingWebhook:
    """HTTP endpoint that receives monitoring alerts and queues retraining jobs for the agent.

    POST /alerts   JSON alert, header X-Signature: sha256=<HMAC of the body>   → 202 queued | 200 logged | 401 | 422
    GET  /jobs     queued and finished jobs
    GET  /health
    """
    REQUIRED = ("alert_id", "rule", "monitor", "severity", "action", "period", "metric", "value", "threshold")

    def __init__(self, secret):
        self.secret, self.queue, self.jobs, self.seen, self.log = secret.encode(), queue.Queue(), {}, set(), []

    def _handle(self, raw, signature):
        expected = "sha256=" + hmac.new(self.secret, raw, hashlib.sha256).hexdigest()
        if not hmac.compare_digest(signature or "", expected):
            return 401, {"accepted": False, "reason": "invalid signature"}
        try:
            alert = json.loads(raw)
        except ValueError:
            return 422, {"accepted": False, "reason": "body is not JSON"}
        missing = [k for k in self.REQUIRED if k not in alert]
        if missing:
            return 422, {"accepted": False, "reason": f"missing fields: {missing}"}
        if alert["alert_id"] in self.seen:
            return 200, {"accepted": False, "reason": "duplicate alert (already received)"}
        self.seen.add(alert["alert_id"])
        if not (alert["severity"] == "critical" and alert["action"] == "retrain"):
            self.log.append({**alert, "routed_to": "logged / owner notified"})
            return 200, {"accepted": True, "routed_to": "logged / owner notified"}
        job = next((j for j in self.jobs.values() if j["period"] == alert["period"] and j["status"] == "queued"), None)
        if job is None:                                # one retraining job per period, however many alerts point at it
            job = {"job_id": "job-" + uuid.uuid4().hex[:6], "period": alert["period"], "status": "queued", "alerts": []}
            self.jobs[job["job_id"]] = job
            self.queue.put(job["job_id"])
        job["alerts"].append(alert)
        self.log.append({**alert, "routed_to": job["job_id"]})
        return 202, {"accepted": True, "routed_to": "retraining agent", "job_id": job["job_id"]}

    def start(self):
        hook = self

        class Handler(BaseHTTPRequestHandler):
            def log_message(self, *args):
                pass

            def _reply(self, code, body):
                data = json.dumps(body, default=str).encode()
                self.send_response(code); self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(data))); self.end_headers(); self.wfile.write(data)

            def do_POST(self):
                if self.path != "/alerts":
                    return self._reply(404, {"error": "not found"})
                raw = self.rfile.read(int(self.headers.get("Content-Length", 0)))
                self._reply(*hook._handle(raw, self.headers.get("X-Signature")))

            def do_GET(self):
                if self.path == "/health":
                    return self._reply(200, {"status": "ok", "queued": hook.queue.qsize()})
                if self.path == "/jobs":
                    return self._reply(200, list(hook.jobs.values()))
                self._reply(404, {"error": "not found"})

        self.server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)       # port 0 = any free port
        threading.Thread(target=self.server.serve_forever, daemon=True).start()
        self.url = f"http://127.0.0.1:{self.server.server_address[1]}"
        return self

    def stop(self):
        self.server.shutdown()


def send_alert(url, alert, secret=WEBHOOK_SECRET):
    """What the monitoring system does: sign the alert and POST it."""
    body = json.dumps(alert, default=str).encode()
    sig = "sha256=" + hmac.new(secret.encode(), body, hashlib.sha256).hexdigest()
    return requests.post(url + "/alerts", data=body, headers={"Content-Type": "application/json", "X-Signature": sig}, timeout=10)

In [ ]:
hook = RetrainingWebhook(WEBHOOK_SECRET).start()
requests.get(hook.url + "/health").json()

In [ ]:
probe_alert = {"alert_id": "test-1", "rule": "smoke test", "monitor": "manual", "severity": "warning", "action": "notify",
               "period": "TEST", "metric": "none", "value": 0, "threshold": 0}
print("signed     →", send_alert(hook.url, probe_alert).json())
print("duplicate  →", send_alert(hook.url, probe_alert).json())
print("forged     →", requests.post(hook.url + "/alerts", json=probe_alert, headers={"X-Signature": "sha256=forged"}).status_code)

In [ ]:
# @title The retraining agent (definition) { display-mode: "form" }
PROMOTE_MARGIN = 0.002


def refit_like_champion(X, y_, name=None):
    """A challenger keeps the live model's recipe and refreshes what it learns from."""
    name = name or CHAMPION
    if name in ROSTER:
        return fit_tfm(name, X, y_)
    if name.endswith("+ LoRA"):
        return fit_tfm(name.replace(" + LoRA", ""), X, y_, strategy="peft", tuning_params={"epochs": LORA_EPOCHS, "learning_rate": 2e-5})
    if name == "XGBoost (tuned)":
        return Tabular(XGB(tree_method="hist", random_state=0, **search.best_params_)).fit(X, y_)
    if name == "HistGradientBoosting":
        return Tabular(Probe(max_iter=300, random_state=0)).fit(X, y_)
    if name == "Linear baseline":
        return Tabular(Linear()).fit(X, y_)
    if name == "TabTune ensemble":
        return TabularEnsemble(models=[{"model_name": m, "tuning_strategy": "inference", "tuning_params": {"device": DEVICE}} for m in members],
                               ensemble_strategy="greedy_selection", task_type=TASK, verbose=False,
                               **({"metric": "r2"} if TASK == "regression" else {})).fit(X, y_)
    return fit_tfm(ROSTER[0], X, y_)


class RetrainingAgent:
    def __init__(self, webhook, monitor):
        self.webhook, self.monitor, self.events = webhook, monitor, []

    def run_pending(self, labelled, next_batch):
        while not self.webhook.queue.empty():
            self.handle(self.webhook.jobs[self.webhook.queue.get()], labelled, next_batch)

    def handle(self, job, labelled, next_batch):
        reasons = [f"{a['rule']} ({a['metric']} {a['value']} ≥ {a['threshold']})" for a in job["alerts"]]
        audit("retrain-agent", f"{job['job_id']} observe", f"{job['period']}: " + "; ".join(reasons))
        if next_batch is None:
            job["status"] = "deferred: no later month to verify on"
            audit("retrain-agent", f"{job['job_id']} decide", "no unseen month to verify a challenger on", "DEFERRED")
            return
        if not human_gate(f"{job['period']}: retrain because {'; '.join(reasons)}?", True):
            job["status"] = "rejected by reviewer"
            return
        recent = pd.concat(labelled, ignore_index=True).dropna(subset=[TARGET_COL])
        keep_old = max(N_CONTEXT - len(recent), N_CONTEXT // 2)
        fresh = pd.concat([ctx.sample(min(keep_old, len(ctx)), random_state=0), recent], ignore_index=True)
        t0 = time.time(); challenger = refit_like_champion(fresh[FEATURES], fresh[TARGET_COL]); secs = time.time() - t0
        yn = next_batch[TARGET_COL]
        c_old, c_new = metric(yn, score_of(self.monitor.model, next_batch[FEATURES])), metric(yn, score_of(challenger, next_batch[FEATURES]))
        audit("retrain-agent", f"{job['job_id']} verify", f"next unseen month: champion {c_old:.4f} vs challenger {c_new:.4f} "
              f"(retrained in {secs:.0f}s on {len(fresh):,} rows)")
        promoted = (c_new - c_old >= PROMOTE_MARGIN) and human_gate(f"Promote the challenger (+{c_new - c_old:.4f} {PRIMARY})?", True)
        if promoted:
            self.monitor.promote(challenger, reference=fresh, version=self.monitor.version + 1)
        audit("retrain-agent", f"{job['job_id']} promote", f"margin required {PROMOTE_MARGIN}",
              f"PROMOTED v{self.monitor.version}" if promoted else "KEPT champion")
        job.update(status="promoted" if promoted else "kept champion", champion_next=c_old, challenger_next=c_new)
        self.events.append({"job_id": job["job_id"], "period": job["period"], "reasons": reasons, "rows": len(fresh), "seconds": secs,
                            "champion_next": c_old, "challenger_next": c_new, "promoted": promoted})

In [ ]:
# @title LLM retraining agent: tools and guardrails { display-mode: "form" }
class LLMRetrainingAgent(RetrainingAgent):
    """Same job as the rule agent. The LLM reads the job, decides whether retraining is justified, and whether to promote; rules guard promotion."""

    def handle(self, job, labelled, next_batch):
        if AGENT_MODE != "llm":
            return super().handle(job, labelled, next_batch)
        st = {}
        reasons = [f"{a['rule']} ({a['metric']} {a['value']} ≥ {a['threshold']})" for a in job["alerts"]]
        keys = ("period", "performance", "max_feature_psi", "share_features_drifted", "prediction_psi", "calibration_error",
                "missing_increase_pp", "out_of_range_pct", "fairness_gap_increase", "model_version")
        def read_job():
            return {"job_id": job["job_id"], "month": job["period"], "metric": PRIMARY, "release_performance": round(float(release_perf), 4),
                    "alerts": [{k: a[k] for k in ("rule", "monitor", "severity", "action", "metric", "value", "threshold")} for a in job["alerts"]],
                    "monitor_history": [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in h.items() if k in keys} for h in self.monitor.history[-4:]],
                    "labelled_months_available": len(labelled), "unseen_next_month_available": next_batch is not None,
                    "promotion_margin": PROMOTE_MARGIN, "live_model": CHAMPION, "live_version": self.monitor.version}
        def retrain_challenger():
            recent = pd.concat(labelled, ignore_index=True).dropna(subset=[TARGET_COL])
            keep_old = max(N_CONTEXT - len(recent), N_CONTEXT // 2)
            fresh = pd.concat([ctx.sample(min(keep_old, len(ctx)), random_state=0), recent], ignore_index=True)
            t0 = time.time(); st["challenger"] = refit_like_champion(fresh[FEATURES], fresh[TARGET_COL])
            st.update(rows=len(fresh), seconds=time.time() - t0, fresh=fresh)
            return {"challenger": f"{CHAMPION} refreshed", "training_rows": len(fresh), "of_which_recent_labelled": len(recent), "seconds": round(st["seconds"], 1)}
        def verify_on_next_month():
            if "challenger" not in st:
                return {"error": "no challenger yet; call retrain_challenger first"}
            if next_batch is None:
                return {"error": "no unseen month to verify on"}
            yn = next_batch[TARGET_COL]
            st["c_old"] = metric(yn, score_of(self.monitor.model, next_batch[FEATURES]))
            st["c_new"] = metric(yn, score_of(st["challenger"], next_batch[FEATURES]))
            return {"metric": PRIMARY, "champion": round(st["c_old"], 4), "challenger": round(st["c_new"], 4),
                    "difference": round(st["c_new"] - st["c_old"], 4), "margin_required": PROMOTE_MARGIN, "clears_margin": st["c_new"] - st["c_old"] >= PROMOTE_MARGIN}
        def promote_challenger():
            if "c_new" not in st:
                return {"error": "verify the challenger first"}
            if st["c_new"] - st["c_old"] < PROMOTE_MARGIN:
                return {"refused": True, "reason": f"challenger does not beat the champion by {PROMOTE_MARGIN} on the unseen month (rule)"}
            self.monitor.promote(st["challenger"], reference=st["fresh"], version=self.monitor.version + 1)
            st["promoted"] = True
            return {"promoted": True, "live_version": self.monitor.version}
        llm = LLMAgent(
            "retraining agent",
            role="You look after a credit model in production. A monitoring webhook has opened a job for you.",
            goal=("Decide whether the alerts justify retraining. If they do, train a challenger, verify it against the champion on the next unseen "
                  "month, and promote it only if it is better by the margin. If retraining is not the right response (for example a data-quality "
                  "problem in the feed, or no unseen month to verify on), finish without retraining and say who should act."),
            rules=["Retraining on a broken feed makes things worse: data-quality alerts are for the data owner.",
                   "Never promote without verifying on the unseen month; promotion needs the margin."],
            tools=[Tool("read_job", read_job, "the alerts, recent monitor history and what is available"),
                   Tool("retrain_challenger", retrain_challenger, "train a challenger with the newest labelled months", approval=True),
                   Tool("verify_on_next_month", verify_on_next_month, "score champion and challenger on the next unseen month"),
                   Tool("promote_challenger", promote_challenger, "make the challenger the live model", approval=True)],
            max_steps=8)
        summary = run_llm_agent(llm, f"Job {job['job_id']} for {job['period']}: {'; '.join(reasons)}.", lambda: RetrainingAgent.handle(self, job, labelled, next_batch))
        if summary == "rule-based fallback":
            return
        if "c_new" in st:
            promoted = bool(st.get("promoted"))
            job.update(status="promoted" if promoted else "kept champion", champion_next=st["c_old"], challenger_next=st["c_new"])
            self.events.append({"job_id": job["job_id"], "period": job["period"], "reasons": reasons, "rows": st["rows"], "seconds": st["seconds"],
                                "champion_next": st["c_old"], "challenger_next": st["c_new"], "promoted": promoted})
        else:
            job["status"] = f"no retraining: {summary[:120]}"

In [ ]:
agent = (LLMRetrainingAgent if AGENT_MODE == "llm" else RetrainingAgent)(hook, monitor)

## 9.4 Six months in production
Each month: the monitor checks the batch, every alert is POSTed to the webhook, and the agent works through whatever the webhook queued.

In [ ]:
for m, batch in enumerate(months):
    period = f"M{m + 1}"
    report = monitor.check(batch, period)                                    # data, prediction, performance, calibration, quality, fairness
    for alert in report["alerts"]:
        send_alert(hook.url, alert)                                          # monitoring system → webhook (signed POST)
    agent.run_pending(labelled=months[: m + 1], next_batch=months[m + 1] if m + 1 < len(months) else None)

In [ ]:
show_agent_trace("retraining agent")

In [ ]:
# @title Monitoring dashboard { display-mode: "form" }
H = pd.DataFrame(monitor.history)
P_ = H.period.tolist()
psi_grid = pd.DataFrame({h["period"]: h["feature_psi"] for h in monitor.history})
ALERTS = pd.DataFrame(monitor.alerts)
rule_of = {r["metric"]: r for r in ALERT_RULES}
fig = make_subplots(rows=3, cols=2, vertical_spacing=0.11, horizontal_spacing=0.12,
                    subplot_titles=("Data drift · PSI by feature and month", "Prediction drift · PSI of the scores",
                                    f"Performance · {PRIMARY} by month", "Calibration", "Data quality", "Alerts raised"))
fig.add_trace(go.Heatmap(z=psi_grid.values, x=psi_grid.columns, y=[c[:22] for c in psi_grid.index], colorscale=[[0, "#F7F7F4"], [0.4, GOLD], [1, RED]],
                         zmin=0, zmax=0.5, colorbar=dict(title="PSI", len=0.3, y=0.88)), 1, 1)
def bands(row, col, metric_):
    r = rule_of[metric_]
    fig.add_hline(y=r["warn"], line_dash="dot", line_color=GOLD, row=row, col=col)
    fig.add_hline(y=r["critical"], line_dash="dot", line_color=RED, row=row, col=col)
fig.add_trace(go.Bar(x=P_, y=H.prediction_psi, marker_color=BLUE, showlegend=False), 1, 2); bands(1, 2, "prediction_psi")
fig.add_trace(go.Scatter(x=P_, y=H.performance, mode="lines+markers", line=dict(color=BLUE, width=3), showlegend=False), 2, 1)
fig.add_hline(y=release_perf, line_dash="dot", line_color=TEAL, row=2, col=1)
for v in H.model_version.drop_duplicates().index[1:]:
    fig.add_vline(x=P_[v], line_dash="dash", line_color=ORANGE, row=2, col=1)
fig.add_trace(go.Scatter(x=P_, y=H.calibration_error, mode="lines+markers", line=dict(color=TEAL, width=3), showlegend=False), 2, 2); bands(2, 2, "calibration_error")
fig.add_trace(go.Bar(x=P_, y=H.missing_increase_pp, name="missing values (+pp)", marker_color=ORANGE), 3, 1)
fig.add_trace(go.Bar(x=P_, y=H.out_of_range_pct, name="out of training range (%)", marker_color=GREY), 3, 1)
if len(ALERTS):
    for sev, col in (("warning", GOLD), ("critical", RED)):
        a = ALERTS[ALERTS.severity == sev]
        fig.add_trace(go.Scatter(x=a.period, y=a.rule, mode="markers", name=sev, marker=dict(size=14, color=col, symbol="square")), 3, 2)
fig.update_layout(template="plotly_white", height=1000, barmode="group", legend=dict(orientation="h", y=-0.05),
                  title=f"Production monitoring · {len(P_)} months · {len(ALERTS)} alerts · live model now v{monitor.version} (dashed line = promotion)")
fig.show()

In [ ]:
# @title What the webhook received, and what the agent did { display-mode: "form" }
log = pd.DataFrame(hook.log)
if len(log):
    display(log[["period", "rule", "monitor", "severity", "value", "threshold", "action", "routed_to"]]
            .style.hide(axis="index").map(lambda v: f"color:{RED};font-weight:600" if v == "critical" else (f"color:{ORANGE}" if v == "warning" else ""), subset=["severity"])
            .set_caption("Webhook log: every alert received and where it was routed"))
display(pd.DataFrame(list(hook.jobs.values()))[["job_id", "period", "status"]].assign(alerts=[len(j["alerts"]) for j in hook.jobs.values()])
        .style.hide(axis="index").set_caption("Retraining jobs") if hook.jobs else Markdown("No retraining job was opened."))
if agent.events:
    E = pd.DataFrame(agent.events)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=E.period + " champion", y=E.champion_next, marker_color=GREY, name="champion", text=E.champion_next.round(4), textposition="outside"))
    fig.add_trace(go.Bar(x=E.period + " challenger", y=E.challenger_next, marker_color=[BLUE if p else "#9DB2F5" for p in E.promoted], name="challenger",
                         text=E.challenger_next.round(4), textposition="outside"))
    lo_ = min(E.champion_next.min(), E.challenger_next.min())
    fig.update_yaxes(range=[lo_ - 0.05 * abs(lo_), max(E.champion_next.max(), E.challenger_next.max()) * 1.03])
    fig.update_layout(template="plotly_white", height=420, title=f"Champion vs challenger on the next unseen month ({PRIMARY}) · dark blue = promoted")
    fig.show()

## 9.5 Regenerated model card
The fact sheet is updated with what monitoring saw and what the agent did, and the card is written again with a change log. The same checks run on the new card.

In [ ]:
# @title Facts from the production period { display-mode: "form" }
H = pd.DataFrame(monitor.history)
ALERTS = pd.DataFrame(monitor.alerts)
changes = []
record("monitoring", months_monitored=len(H), alerts_raised=len(ALERTS), critical_alerts=int((ALERTS.severity == "critical").sum()) if len(ALERTS) else 0,
       drift_feature=drift_feature, max_feature_psi=float(H.max_feature_psi.max()), worst_month_performance=float(H.performance.min()),
       final_month_performance=float(H.performance.iloc[-1]), release_performance=release_perf, data_quality_incident_field=QUALITY_FIELD,
       alert_routing="critical alerts POSTed to a signed retraining webhook; warnings notify the owner; data-quality alerts go to the data owner",
       alert_thresholds="; ".join(f"{r['rule']}: warning {r['warn']}, critical {r['critical']}" for r in ALERT_RULES), promotion_margin=PROMOTE_MARGIN)
E = pd.DataFrame(agent.events)
if len(E):
    record("retraining", retraining_runs=len(E), promotions=int(E.promoted.sum()), first_trigger_month=E.period.iloc[0],
           outcome=f"{len(E)} retraining run(s), {int(E.promoted.sum())} promoted", live_model_version=f"v{monitor.version}",
           human_approvals="retraining and promotion gates")
    for e in agent.events:
        changes += [f"{e['period']}: critical alerts ({'; '.join(e['reasons'])}) opened {e['job_id']}; challenger trained on {e['rows']} rows.",
                    f"{e['period']}: on the next unseen month the champion scored {e['champion_next']:.4f} and the challenger {e['challenger_next']:.4f}; "
                    + ("challenger promoted after human approval." if e["promoted"] else "champion kept (promotion margin not met).")]
else:
    record("retraining", outcome="no retraining was triggered during the monitoring period", live_model_version=f"v{monitor.version}")
    changes = ["Monitoring period completed with no retraining."]
FACTS["use_case"]["version"] = f"v{monitor.version}"
for c in changes:
    print("•", c)

In [ ]:
CARD_V2, TRACE_V2, CHECKS_V2 = generate_model_card(FACTS, changes=changes, version_label="v2")
show_card(CARD_V2, TRACE_V2, CHECKS_V2)
open("model_card_v2.md", "w").write(CARD_V2);

# 10 · Conclusions

## Agent activity
What each agent did in this run: every LLM call (through the Lexsi gateway, so each is also traced in the Lexsi project), the tools the agents called, and which actions a human approved.

In [ ]:
# @title Agent activity: LLM calls, tools and approvals { display-mode: "form" }
S, C_ = pd.DataFrame(AGENT_STEPS), pd.DataFrame(AGENT_CALLS)
if not len(S):
    print(f"The agents ran in rule mode (AGENT_MODE = '{AGENT_MODE}'): no LLM calls.")
else:
    by = C_.groupby("agent").agg(calls=("agent", "size"), seconds=("seconds", "sum"), input_tokens=("input_tokens", "sum"), output_tokens=("output_tokens", "sum"))
    tools_ = S.groupby(["agent", "tool"]).size().rename("calls").reset_index()
    fig = make_subplots(rows=1, cols=3, column_widths=[0.28, 0.28, 0.44], subplot_titles=("LLM calls per agent", "Tokens per agent", "Tools called"))
    fig.add_trace(go.Bar(x=by.index, y=by.calls, marker_color=BLUE, showlegend=False, text=by.calls, textposition="outside"), 1, 1)
    fig.add_trace(go.Bar(x=by.index, y=by.input_tokens, name="input", marker_color=GREY), 1, 2)
    fig.add_trace(go.Bar(x=by.index, y=by.output_tokens, name="output", marker_color=ORANGE), 1, 2)
    for j, (a_, g_) in enumerate(tools_.groupby("agent")):
        fig.add_trace(go.Bar(y=g_.tool, x=g_.calls, orientation="h", name=a_, marker_color=PALETTE[j % len(PALETTE)]), 1, 3)
    fig.update_layout(template="plotly_white", height=420, barmode="stack",
                      title=f"Agent activity · {len(C_)} LLM calls · {int(C_.input_tokens.sum() + C_.output_tokens.sum()):,} tokens · {C_.model.iloc[0]}")
    fig.show()
    ap = S.dropna(subset=["approved"])
    display(pd.DataFrame({"steps": S.groupby("agent").size(), "actions needing approval": ap.groupby("agent").size(),
                          "approved": ap[ap.approved == True].groupby("agent").size()}).fillna(0).astype(int))
record("governance", agents="LLM agents through the Lexsi gateway" if AGENT_MODE == "llm" else "rule-based agents",
       agent_llm_calls=len(AGENT_CALLS), agent_actions_approved=int(sum(1 for s in AGENT_STEPS if s["approved"] is True)))

In [ ]:
# @title Build the plain-English conclusions from the recorded facts { display-mode: "form" }
P, S, U = FACTS["performance"], FACTS["selection"], FACTS["uncertainty"]
headline = f"{P['roc_auc']:.3f} ROC AUC (F1 {P['f1']:.3f} at a threshold of {P['threshold']:.2f})" if TASK == "classification" else f"R² {P['r2']:.3f}, typical error {P['mape_pct']:.0f}%, {P['within_20pct_share']:.0f}% of recommendations within 20% of the actual amount"
CONCLUSIONS = [
    f"**Data:** {FACTS['preparation']['features_before']} candidate features were reduced to {FACTS['features']['final_feature_count']} by a data preparation agent, with a human approval at every step.",
    f"**Model:** {S['candidates_succeeded']} models were raced; **{S['selected_model']}** was selected on validation data ({S['selection_metric']} {S['selected_validation_score']:.4f}; best classic model {S['best_classic_validation_score']:.4f}).",
    f"**Performance on unseen data:** {headline}.",
    f"**Uncertainty:** a {U['target_coverage']:.0%} coverage promise measured {U['measured_coverage']:.1%} on the test set.",
    f"**Fairness:** measured across {FACTS['fairness']['groups']} {FACTS['fairness']['attribute']} groups; the gaps are in section 7.2.",
    f"**Documentation:** the model card was written by {TRACE_V2['writer']} from a 14-section standard; {CHECKS_V2['grounded_pct']:.0f}% of its numbers trace back to recorded facts, with {CHECKS_V2['citations']} citations checked automatically.",
    f"**Production:** over {MONTHS} simulated months five monitors raised {FACTS['monitoring']['alerts_raised']} alerts ({FACTS['monitoring']['critical_alerts']} critical); critical alerts reached the retraining agent through a signed webhook; outcome: {FACTS['retraining']['outcome']}.",
    "**Next step:** the same workflow runs on the Lexsi platform with managed compute, a model registry and case-level audit.",
]

In [ ]:
# @title Conclusions in plain English, and the full audit trail { display-mode: "form" }
display(Markdown("## What this exercise showed\n\n" + "\n".join(f"- {line}" for line in CONCLUSIONS)))
trail = pd.DataFrame(AUDIT)
if len(trail):
    fig = px.histogram(trail, x="step", color="actor", color_discrete_sequence=PALETTE, title="Audit trail · logged agent steps and human decisions")
    fig.update_layout(template="plotly_white", height=360, xaxis_title=""); fig.show()
    display(trail.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))